データサイエンス100本ノック（構造化データ加工編） - C # 
--

## はじめに
- 初めに以下のセルを実行してください
- 必要なライブラリのインポートとCSVファイルからのデータ読み込みを行います
- 利用が想定されるライブラリは以下セルでインポートしています
- その他利用したいライブラリがあれば適宜インストールしてください（"#r "nuget:ライブラリ名" で nuget パッケージがインストール可能）
- 処理は複数回に分けても構いません
- 名前、住所等はダミーデータであり、実在するものではありません

In [1]:
dotnet --version

9.0.305


In [1]:
#!about

.NET Interactive© 2020 Microsoft CorporationVersion: 1.0.456201+e5dc02dbdc933d895d722e69bfa98d580dcda8d9Library version: 1.0.0-beta.23562.1+e5dc02dbdc933d895d722e69bfa98d580dcda8d9Build date: 2023-11-29T14:04:54.2696190Zhttps://github.com/dotnet/interactive


In [1]:
// 必要なライブラリを nuget パッケージからインストールします。
#r "nuget:CsvHelper, 19.0.0"
#r "nuget:MathNet.Numerics"
#r "nuget: Plotly.NET.Interactive"
#r "nuget: Plotly.NET.CSharp"
#r "nuget: Plotly.NET"



Installed Packages CsvHelper, 19.0.0 MathNet.Numerics, 5.0.0 Plotly.NET, 5.1.0 Plotly.NET.CSharp, 0.13.0 Plotly.NET.Interactive, 5.0.0

Loading extensions from `C:\Users\kazuh\.nuget\packages\plotly.net.interactive\5.0.0\lib\netstandard2.1\Plotly.NET.Interactive.dll`

In [2]:
// using 設定
using System.Text.RegularExpressions;
using CsvHelper.Configuration.Attributes;
using MathNet.Numerics;
using Microsoft.DotNet.Interactive.Formatting;
using static Microsoft.DotNet.Interactive.Formatting.PocketViewTags;
using Microsoft.AspNetCore.Html;

// 日付表示のキレイ化
Formatter.Register<DateTime>((date, writer) => 
{
    if (date.Hour == 0 && date.Minute == 0 && date.Second == 0) {
        writer.Write(span(date.ToString("yyyy-MM-dd")));
    } else {
        writer.Write(span(date.ToString("yyyy-MM-dd HH:mm:ss")));
    }
}, "text/html");

In [3]:
// データを CSV ファイルから取得する定義＆読み込み
using CsvHelper.Configuration.Attributes;
using System.IO;
using System.Globalization;
using CsvHelper;

static List<T> LoadFromCsvFile<T>(string fileName) 
{
    using (var reader = new StreamReader(fileName))
    using (var csv = new CsvReader(reader, CultureInfo.InvariantCulture))
    {
        try {
            csv.Configuration.HasHeaderRecord = true;
            var records = csv.GetRecords<T>();
            return records.ToList();
        } catch(Exception exp) {
            throw new Exception("file=" + fileName, exp);
        }
    }
}
public class Category
{
    public string category_major_cd { get; set; }
    public string category_major_name { get; set; }
    public string category_medium_cd { get; set; }
    public string category_medium_name { get; set; }
    public string category_small_cd { get; set; }
    public string category_small_name { get; set; }
}

public class Customer
{
    public string customer_id { get; set; }
    public string customer_name { get; set; }
    public string gender_cd { get; set; }
    public string gender { get; set; }
    public DateTime? birth_day { get; set; }
    public int? age { get; set; }
    public string postal_cd { get; set; }
    public string address { get; set; }
    public string application_store_cd { get; set; }
    public string application_date { get; set; }
    public string status_cd { get; set; }
}

public class Geocode
{
    public string postal_cd { get; set; }
    public string prefecture { get; set; }
    public string city { get; set; }
    public string town { get; set; }
    public string street { get; set; }
    public string address { get; set; }
    public string full_address { get; set; }
    public decimal? longitude { get; set; }
    [Name(" latitude")]
    public decimal? latitude { get; set; }
}
// ※注：CSV ファイルの項目定義に合わせて " latitude" と修正

public class Product
{
    public string product_cd { get; set; }
    public string category_major_cd { get; set; }
    public string category_medium_cd { get; set; }
    public string category_small_cd { get; set; }
    public int? unit_price { get; set; }
    public int? unit_cost { get; set; }
}

public class Receipt
{
    public string sales_ymd { get; set; }
    public int? sales_epoch { get; set; }
    public string store_cd { get; set; }
    public short receipt_no { get; set; }
    public short receipt_sub_no { get; set; }
    public string customer_id { get; set; }
    public string product_cd { get; set; }
    public int? quantity { get; set; }
    public int? amount { get; set; }
}

public class Store
{
    public string store_cd { get; set; }
    public string store_name { get; set; }
    public string prefecture_cd { get; set; }
    public string prefecture { get; set; }
    public string address { get; set; }
    public string address_kana { get; set; }
    public string tel_no { get; set; }
    public decimal ?longitude { get; set; }
    public decimal ?latitude { get; set; }
    public decimal ?floor_area { get; set; }
}

var df_customer = LoadFromCsvFile<Customer>("./customer.csv");
var df_category = LoadFromCsvFile<Category>("./category.csv");
var df_product = LoadFromCsvFile<Product>("./product.csv");
var df_receipt = LoadFromCsvFile<Receipt>("./receipt.csv");
var df_store = LoadFromCsvFile<Store>("./store.csv");
var df_geocode = LoadFromCsvFile<Geocode>("./geocode.csv");

# 演習問題

---
> P-001: レシート明細のデータフレーム（df_receipt）から全項目の先頭10件を表示し、どのようなデータを保有しているか目視で確認せよ。

In [7]:
df_receipt.Take(5).DisplayTable()


sales_ymd,sales_epoch,store_cd,receipt_no,receipt_sub_no,customer_id,product_cd,quantity,amount
20181103,1541203200,S14006,112,1,CS006214000001,P070305012,1,158
20181118,1542499200,S13008,1132,2,CS008415000097,P070701017,1,81
20170712,1499817600,S14028,1102,1,CS028414000014,P060101005,1,170
20190205,1549324800,S14042,1132,1,ZZ000000000000,P050301001,1,25
20180821,1534809600,S14025,1102,2,CS025415000050,P060102007,1,90


---
> P-002: レシート明細のデータフレーム（df_receipt）から売上日（sales_ymd）、顧客ID（customer_id）、商品コード（product_cd）、売上金額（amount）の順に列を指定し、10件表示させよ。

In [7]:
df_receipt.Select(r => new { r.sales_ymd, r.customer_id, r.product_cd, r.amount }).Take(10).DisplayTable()

sales_ymd,customer_id,product_cd,amount
20181103,CS006214000001,P070305012,158
20181118,CS008415000097,P070701017,81
20170712,CS028414000014,P060101005,170
20190205,ZZ000000000000,P050301001,25
20180821,CS025415000050,P060102007,90
20190605,CS003515000195,P050102002,138
20181205,CS024514000042,P080101005,30
20190922,CS040415000178,P070501004,128
20170504,ZZ000000000000,P071302010,770
20191010,CS027514000015,P071101003,680


---
> P-003: レシート明細のデータフレーム（df_receipt）から売上日（sales_ymd）、顧客ID（customer_id）、商品コード（product_cd）、売上金額（amount）の順に列を指定し、10件表示させよ。ただし、sales_ymdはsales_dateに項目名を変更しながら抽出すること。

In [7]:
df_receipt.Select(r => new {
    sales_date = r.sales_ymd,
    r.customer_id,
    r.product_cd, 
    r.amount
}).Take(10).DisplayTable()

sales_date,customer_id,product_cd,amount
20181103,CS006214000001,P070305012,158
20181118,CS008415000097,P070701017,81
20170712,CS028414000014,P060101005,170
20190205,ZZ000000000000,P050301001,25
20180821,CS025415000050,P060102007,90
20190605,CS003515000195,P050102002,138
20181205,CS024514000042,P080101005,30
20190922,CS040415000178,P070501004,128
20170504,ZZ000000000000,P071302010,770
20191010,CS027514000015,P071101003,680


---
> P-004: レシート明細のデータフレーム（df_receipt）から売上日（sales_ymd）、顧客ID（customer_id）、商品コード（product_cd）、売上金額（amount）の順に列を指定し、以下の条件を満たすデータを抽出せよ。
> - 顧客ID（customer_id）が"CS018205000001"

In [8]:
df_receipt.Select(r => new {r.sales_ymd, r.customer_id, r.product_cd, r.amount})
    .Where(r => r.customer_id == "CS018205000001").DisplayTable()   

sales_ymd,customer_id,product_cd,amount
20180911,CS018205000001,P071401012,2200
20180414,CS018205000001,P060104007,600
20170614,CS018205000001,P050206001,990
20170614,CS018205000001,P060702015,108
20190216,CS018205000001,P071005024,102
20180414,CS018205000001,P071101002,278
20190226,CS018205000001,P070902035,168
20190924,CS018205000001,P060805001,495
20190226,CS018205000001,P071401020,2200
20180911,CS018205000001,P071401005,1100


---
> P-005: レシート明細のデータフレーム（df_receipt）から売上日（sales_ymd）、顧客ID（customer_id）、商品コード（product_cd）、売上金額（amount）の順に列を指定し、以下の条件を満たすデータを抽出せよ。
> - 顧客ID（customer_id）が"CS018205000001"
> - 売上金額（amount）が1,000以上

In [9]:
df_receipt.Select(r => new {r.sales_ymd, r.customer_id, r.product_cd, r.amount})
    .Where(r => r.customer_id == "CS018205000001" && r.amount >= 1000)
    .DisplayTable()

sales_ymd,customer_id,product_cd,amount
20180911,CS018205000001,P071401012,2200
20190226,CS018205000001,P071401020,2200
20180911,CS018205000001,P071401005,1100


---
> P-006: レシート明細データフレーム「df_receipt」から売上日（sales_ymd）、顧客ID（customer_id）、商品コード（product_cd）、売上数量（quantity）、売上金額（amount）の順に列を指定し、以下の条件を満たすデータを抽出せよ。
> - 顧客ID（customer_id）が"CS018205000001"
> - 売上金額（amount）が1,000以上または売上数量（quantity）が5以上

In [10]:
df_receipt.Select(r => new {r.sales_ymd, r.customer_id, r.product_cd, r.amount, r.quantity})
    .Where(r => r.customer_id == "CS018205000001" && (r.amount >= 1000 || r.quantity >= 5))
    .DisplayTable()

sales_ymd,customer_id,product_cd,amount,quantity
20180911,CS018205000001,P071401012,2200,1
20180414,CS018205000001,P060104007,600,6
20170614,CS018205000001,P050206001,990,5
20190226,CS018205000001,P071401020,2200,1
20180911,CS018205000001,P071401005,1100,1


---
> P-007: レシート明細のデータフレーム（df_receipt）から売上日（sales_ymd）、顧客ID（customer_id）、商品コード（product_cd）、売上金額（amount）の順に列を指定し、以下の条件を満たすデータを抽出せよ。
> - 顧客ID（customer_id）が"CS018205000001"
> - 売上金額（amount）が1,000以上2,000以下

In [11]:
df_receipt.Select(r => new {r.sales_ymd, r.customer_id, r.product_cd, r.amount})
    .Where(r => r.customer_id == "CS018205000001" && (1000 <= r.amount && r.amount <= 2000))
    .DisplayTable()

sales_ymd,customer_id,product_cd,amount
20180911,CS018205000001,P071401005,1100


---
> P-008: レシート明細のデータフレーム（df_receipt）から売上日（sales_ymd）、顧客ID（customer_id）、商品コード（product_cd）、売上金額（amount）の順に列を指定し、以下の条件を満たすデータを抽出せよ。
> - 顧客ID（customer_id）が"CS018205000001"
> - 商品コード（product_cd）が"P071401019"以外

In [12]:
df_receipt.Select(r => new {r.sales_ymd, r.customer_id, r.product_cd, r.amount})
    .Where(r => r.customer_id == "CS018205000001" && r.product_cd != "P071401019")
    .DisplayTable()

sales_ymd,customer_id,product_cd,amount
20180911,CS018205000001,P071401012,2200
20180414,CS018205000001,P060104007,600
20170614,CS018205000001,P050206001,990
20170614,CS018205000001,P060702015,108
20190216,CS018205000001,P071005024,102
20180414,CS018205000001,P071101002,278
20190226,CS018205000001,P070902035,168
20190924,CS018205000001,P060805001,495
20190226,CS018205000001,P071401020,2200
20180911,CS018205000001,P071401005,1100


---
> P-009: 以下の処理において、出力結果を変えずにORをANDに書き換えよ。

`df_store.Where( r => !(r.prefecture_cd == "13" || r.floor_area > 900))`

In [13]:
df_store.Where( r => r.prefecture_cd != "13" && r.floor_area <= 900)
.DisplayTable()

store_cd,store_name,prefecture_cd,prefecture,address,address_kana,tel_no,longitude,latitude,floor_area
S14046,北山田店,14,神奈川県,神奈川県横浜市都筑区北山田一丁目,カナガワケンヨコハマシツヅキクキタヤマタイッチョウメ,045-123-4049,139.5916,35.56189,831.0
S14011,日吉本町店,14,神奈川県,神奈川県横浜市港北区日吉本町四丁目,カナガワケンヨコハマシコウホククヒヨシホンチョウヨンチョウメ,045-123-4033,139.6316,35.54655,890.0
S12013,習志野店,12,千葉県,千葉県習志野市芝園一丁目,チバケンナラシノシシバゾノイッチョウメ,047-123-4002,140.022,35.66122,808.0


---
> P-010: 店舗データフレーム（df_store）から、店舗コード（store_cd）が"S14"で始まるものだけ全項目抽出し、10件だけ表示せよ。

In [14]:
df_store.Where(r => r.store_cd.StartsWith("S14")).Take(10).DisplayTable()

store_cd,store_name,prefecture_cd,prefecture,address,address_kana,tel_no,longitude,latitude,floor_area
S14010,菊名店,14,神奈川県,神奈川県横浜市港北区菊名一丁目,カナガワケンヨコハマシコウホククキクナイッチョウメ,045-123-4032,139.6326,35.50049,1732.0
S14033,阿久和店,14,神奈川県,神奈川県横浜市瀬谷区阿久和西一丁目,カナガワケンヨコハマシセヤクアクワニシイッチョウメ,045-123-4043,139.4961,35.45918,1495.0
S14036,相模原中央店,14,神奈川県,神奈川県相模原市中央二丁目,カナガワケンサガミハラシチュウオウニチョウメ,042-123-4045,139.3716,35.57327,1679.0
S14040,長津田店,14,神奈川県,神奈川県横浜市緑区長津田みなみ台五丁目,カナガワケンヨコハマシミドリクナガツタミナミダイゴチョウメ,045-123-4046,139.4994,35.52398,1548.0
S14050,阿久和西店,14,神奈川県,神奈川県横浜市瀬谷区阿久和西一丁目,カナガワケンヨコハマシセヤクアクワニシイッチョウメ,045-123-4053,139.4961,35.45918,1830.0
S14028,二ツ橋店,14,神奈川県,神奈川県横浜市瀬谷区二ツ橋町,カナガワケンヨコハマシセヤクフタツバシチョウ,045-123-4042,139.4963,35.46304,1574.0
S14012,本牧和田店,14,神奈川県,神奈川県横浜市中区本牧和田,カナガワケンヨコハマシナカクホンモクワダ,045-123-4034,139.6582,35.42156,1341.0
S14046,北山田店,14,神奈川県,神奈川県横浜市都筑区北山田一丁目,カナガワケンヨコハマシツヅキクキタヤマタイッチョウメ,045-123-4049,139.5916,35.56189,831.0
S14022,逗子店,14,神奈川県,神奈川県逗子市逗子一丁目,カナガワケンズシシズシイッチョウメ,046-123-4036,139.5789,35.29642,1838.0
S14011,日吉本町店,14,神奈川県,神奈川県横浜市港北区日吉本町四丁目,カナガワケンヨコハマシコウホククヒヨシホンチョウヨンチョウメ,045-123-4033,139.6316,35.54655,890.0


---
> P-011: 顧客データフレーム（df_customer）から顧客ID（customer_id）の末尾が1のものだけ全項目抽出し、10件だけ表示せよ。

In [15]:
df_customer.Where(r => r.customer_id.EndsWith("1")).Take(10).DisplayTable()

customer_id,customer_name,gender_cd,gender,birth_day,age,postal_cd,address,application_store_cd,application_date,status_cd
CS037613000071,六角 雅彦,9,不明,1952-04-01,66,136-0076,東京都江東区南砂**********,S13037,20150414,0-00000000-0
CS028811000001,堀井 かおり,1,女性,1933-03-27,86,245-0016,神奈川県横浜市泉区和泉町**********,S14028,20160115,0-00000000-0
CS040412000191,川井 郁恵,1,女性,1977-01-05,42,226-0021,神奈川県横浜市緑区北八朔町**********,S14040,20151101,1-20091025-4
CS028314000011,小菅 あおい,1,女性,1983-11-26,35,246-0038,神奈川県横浜市瀬谷区宮沢**********,S14028,20151123,1-20080426-5
CS039212000051,藤島 恵梨香,1,女性,1997-02-03,22,166-0001,東京都杉並区阿佐谷北**********,S13039,20171121,1-20100215-4
CS015412000111,松居 奈月,1,女性,1972-10-04,46,136-0071,東京都江東区亀戸**********,S13015,20150629,0-00000000-0
CS004702000041,野島 洋,0,男性,1943-08-24,75,176-0022,東京都練馬区向山**********,S13004,20170218,0-00000000-0
CS041515000001,栗田 千夏,1,女性,1967-01-02,52,206-0001,東京都多摩市和田**********,S13041,20160422,E-20100803-F
CS029313000221,北条 ひかり,1,女性,1987-06-19,31,279-0011,千葉県浦安市美浜**********,S12029,20180810,0-00000000-0
CS034312000071,望月 奈央,1,女性,1980-09-20,38,213-0026,神奈川県川崎市高津区久末**********,S14034,20160106,0-00000000-0


---
> P-012: 店舗データフレーム（df_store）から横浜市の店舗だけ全項目表示せよ。

In [16]:
df_store.Where(r => r.address.Contains("横浜市")).DisplayTable()

store_cd,store_name,prefecture_cd,prefecture,address,address_kana,tel_no,longitude,latitude,floor_area
S14010,菊名店,14,神奈川県,神奈川県横浜市港北区菊名一丁目,カナガワケンヨコハマシコウホククキクナイッチョウメ,045-123-4032,139.6326,35.50049,1732.0
S14033,阿久和店,14,神奈川県,神奈川県横浜市瀬谷区阿久和西一丁目,カナガワケンヨコハマシセヤクアクワニシイッチョウメ,045-123-4043,139.4961,35.45918,1495.0
S14040,長津田店,14,神奈川県,神奈川県横浜市緑区長津田みなみ台五丁目,カナガワケンヨコハマシミドリクナガツタミナミダイゴチョウメ,045-123-4046,139.4994,35.52398,1548.0
S14050,阿久和西店,14,神奈川県,神奈川県横浜市瀬谷区阿久和西一丁目,カナガワケンヨコハマシセヤクアクワニシイッチョウメ,045-123-4053,139.4961,35.45918,1830.0
S14028,二ツ橋店,14,神奈川県,神奈川県横浜市瀬谷区二ツ橋町,カナガワケンヨコハマシセヤクフタツバシチョウ,045-123-4042,139.4963,35.46304,1574.0
S14012,本牧和田店,14,神奈川県,神奈川県横浜市中区本牧和田,カナガワケンヨコハマシナカクホンモクワダ,045-123-4034,139.6582,35.42156,1341.0
S14046,北山田店,14,神奈川県,神奈川県横浜市都筑区北山田一丁目,カナガワケンヨコハマシツヅキクキタヤマタイッチョウメ,045-123-4049,139.5916,35.56189,831.0
S14011,日吉本町店,14,神奈川県,神奈川県横浜市港北区日吉本町四丁目,カナガワケンヨコハマシコウホククヒヨシホンチョウヨンチョウメ,045-123-4033,139.6316,35.54655,890.0
S14048,中川中央店,14,神奈川県,神奈川県横浜市都筑区中川中央二丁目,カナガワケンヨコハマシツヅキクナカガワチュウオウニチョウメ,045-123-4051,139.5758,35.54912,1657.0
S14042,新山下店,14,神奈川県,神奈川県横浜市中区新山下二丁目,カナガワケンヨコハマシナカクシンヤマシタニチョウメ,045-123-4047,139.6593,35.43894,1044.0


---
> P-013: 顧客データフレーム（df_customer）から、ステータスコード（status_cd）の先頭がアルファベットのA〜Fで始まるデータを全項目抽出し、10件だけ表示せよ。

In [17]:
df_customer.Where(r => Regex.IsMatch(r.status_cd, "^[A-F]")).Take(10).DisplayTable()

customer_id,customer_name,gender_cd,gender,birth_day,age,postal_cd,address,application_store_cd,application_date,status_cd
CS031415000172,宇多田 貴美子,1,女性,1976-10-04,42,151-0053,東京都渋谷区代々木**********,S13031,20150529,D-20100325-C
CS015414000103,奥野 陽子,1,女性,1977-08-09,41,136-0073,東京都江東区北砂**********,S13015,20150722,B-20100609-B
CS011215000048,芦田 沙耶,1,女性,1992-02-01,27,223-0062,神奈川県横浜市港北区日吉本町**********,S14011,20150228,C-20100421-9
CS029415000023,梅田 里穂,1,女性,1976-01-17,43,279-0043,千葉県浦安市富士見**********,S12029,20150610,D-20100918-E
CS035415000029,寺沢 真希,9,不明,1977-09-27,41,158-0096,東京都世田谷区玉川台**********,S13035,20141220,F-20101029-F
CS031415000106,宇野 由美子,1,女性,1970-02-26,49,151-0053,東京都渋谷区代々木**********,S13031,20150201,F-20100511-E
CS029215000025,石倉 美帆,1,女性,1993-09-28,25,279-0022,千葉県浦安市今川**********,S12029,20150708,B-20100820-C
CS033605000005,猪股 雄太,0,男性,1955-12-05,63,246-0031,神奈川県横浜市瀬谷区瀬谷**********,S14033,20150425,F-20100917-E
CS033415000229,板垣 菜々美,1,女性,1977-11-07,41,246-0021,神奈川県横浜市瀬谷区二ツ橋町**********,S14033,20150712,F-20100326-E
CS008415000145,黒谷 麻緒,1,女性,1977-06-27,41,157-0067,東京都世田谷区喜多見**********,S13008,20150829,F-20100622-F


---
> P-014: 顧客データフレーム（df_customer）から、ステータスコード（status_cd）の末尾が数字の1〜9で終わるデータを全項目抽出し、10件だけ表示せよ。

In [18]:
df_customer.Where(r => Regex.IsMatch(r.status_cd, "[1-9]$")).Take(10).DisplayTable()

customer_id,customer_name,gender_cd,gender,birth_day,age,postal_cd,address,application_store_cd,application_date,status_cd
CS001215000145,田崎 美紀,1,女性,1995-03-29,24,144-0055,東京都大田区仲六郷**********,S13001,20170605,6-20090929-2
CS033513000180,安斎 遥,1,女性,1962-07-11,56,241-0823,神奈川県横浜市旭区善部町**********,S14033,20150728,6-20080506-5
CS011215000048,芦田 沙耶,1,女性,1992-02-01,27,223-0062,神奈川県横浜市港北区日吉本町**********,S14011,20150228,C-20100421-9
CS040412000191,川井 郁恵,1,女性,1977-01-05,42,226-0021,神奈川県横浜市緑区北八朔町**********,S14040,20151101,1-20091025-4
CS009315000023,皆川 文世,1,女性,1980-04-15,38,154-0012,東京都世田谷区駒沢**********,S13009,20150319,5-20080322-1
CS015315000033,福士 璃奈子,1,女性,1983-03-17,36,135-0043,東京都江東区塩浜**********,S13015,20141024,4-20080219-3
CS023513000066,神戸 そら,1,女性,1961-12-17,57,210-0005,神奈川県川崎市川崎区東田町**********,S14023,20150915,5-20100524-9
CS035513000134,市川 美帆,1,女性,1960-03-27,59,156-0053,東京都世田谷区桜**********,S13035,20150227,8-20100711-9
CS001515000263,高松 夏空,1,女性,1962-11-09,56,144-0051,東京都大田区西蒲田**********,S13001,20160812,1-20100804-1
CS040314000027,鶴田 きみまろ,9,不明,1986-03-26,33,226-0027,神奈川県横浜市緑区長津田**********,S14040,20150122,2-20080426-4


---
> P-015: 顧客データフレーム（df_customer）から、ステータスコード（status_cd）の先頭がアルファベットのA〜Fで始まり、末尾が数字の1〜9で終わるデータを全項目抽出し、10件だけ表示せよ。

In [19]:
df_customer.Where(r => Regex.IsMatch(r.status_cd, "^[A-F].*[1-9]$")).Take(10).DisplayTable()

customer_id,customer_name,gender_cd,gender,birth_day,age,postal_cd,address,application_store_cd,application_date,status_cd
CS011215000048,芦田 沙耶,1,女性,1992-02-01,27,223-0062,神奈川県横浜市港北区日吉本町**********,S14011,20150228,C-20100421-9
CS022513000105,島村 貴美子,1,女性,1962-03-12,57,249-0002,神奈川県逗子市山の根**********,S14022,20150320,A-20091115-7
CS001515000096,水野 陽子,9,不明,1960-11-29,58,144-0053,東京都大田区蒲田本町**********,S13001,20150614,A-20100724-7
CS013615000053,西脇 季衣,1,女性,1953-10-18,65,261-0026,千葉県千葉市美浜区幕張西**********,S12013,20150128,B-20100329-6
CS020412000161,小宮 薫,1,女性,1974-05-21,44,174-0042,東京都板橋区東坂下**********,S13020,20150822,B-20081021-3
CS001215000097,竹中 あさみ,1,女性,1990-07-25,28,146-0095,東京都大田区多摩川**********,S13001,20170315,A-20100211-2
CS035212000007,内村 恵梨香,1,女性,1990-12-04,28,152-0023,東京都目黒区八雲**********,S13035,20151013,B-20101018-6
CS002515000386,野田 コウ,1,女性,1963-05-30,55,185-0013,東京都国分寺市西恋ケ窪**********,S13002,20160410,C-20100127-8
CS001615000372,稲垣 寿々花,1,女性,1956-10-29,62,144-0035,東京都大田区南蒲田**********,S13001,20170403,A-20100104-1
CS032512000121,松井 知世,1,女性,1962-09-04,56,210-0011,神奈川県川崎市川崎区富士見**********,S13032,20150727,A-20100103-5


---
> P-016: 店舗データフレーム（df_store）から、電話番号（tel_no）が3桁-3桁-4桁のデータを全項目表示せよ。

In [23]:

Formatter.ListExpansionLimit = int.MaxValue; // 全データを表示するために制限解除

df_store.Where(r => Regex.IsMatch(r.tel_no, "^[0-9]{3}-[0-9]{3}-[0-9]{4}$")).DisplayTable();
    
Formatter.ListExpansionLimit = 20; // データ表示件数の制限


store_cd,store_name,prefecture_cd,prefecture,address,address_kana,tel_no,longitude,latitude,floor_area
S12014,千草台店,12,千葉県,千葉県千葉市稲毛区千草台一丁目,チバケンチバシイナゲクチグサダイイッチョウメ,043-123-4003,140.118,35.63559,1698.0
S13002,国分寺店,13,東京都,東京都国分寺市本多二丁目,トウキョウトコクブンジシホンダニチョウメ,042-123-4008,139.4802,35.70566,1735.0
S14010,菊名店,14,神奈川県,神奈川県横浜市港北区菊名一丁目,カナガワケンヨコハマシコウホククキクナイッチョウメ,045-123-4032,139.6326,35.50049,1732.0
S14033,阿久和店,14,神奈川県,神奈川県横浜市瀬谷区阿久和西一丁目,カナガワケンヨコハマシセヤクアクワニシイッチョウメ,045-123-4043,139.4961,35.45918,1495.0
S14036,相模原中央店,14,神奈川県,神奈川県相模原市中央二丁目,カナガワケンサガミハラシチュウオウニチョウメ,042-123-4045,139.3716,35.57327,1679.0
S14040,長津田店,14,神奈川県,神奈川県横浜市緑区長津田みなみ台五丁目,カナガワケンヨコハマシミドリクナガツタミナミダイゴチョウメ,045-123-4046,139.4994,35.52398,1548.0
S14050,阿久和西店,14,神奈川県,神奈川県横浜市瀬谷区阿久和西一丁目,カナガワケンヨコハマシセヤクアクワニシイッチョウメ,045-123-4053,139.4961,35.45918,1830.0
S13052,森野店,13,東京都,東京都町田市森野三丁目,トウキョウトマチダシモリノサンチョウメ,042-123-4030,139.4383,35.55293,1087.0
S14028,二ツ橋店,14,神奈川県,神奈川県横浜市瀬谷区二ツ橋町,カナガワケンヨコハマシセヤクフタツバシチョウ,045-123-4042,139.4963,35.46304,1574.0
S14012,本牧和田店,14,神奈川県,神奈川県横浜市中区本牧和田,カナガワケンヨコハマシナカクホンモクワダ,045-123-4034,139.6582,35.42156,1341.0


---
> P-17: 顧客データフレーム（df_customer）を生年月日（birth_day）で高齢順にソートし、先頭10件を全項目表示せよ。

In [24]:
df_customer.OrderBy(r => r.birth_day).Take(10).DisplayTable()

customer_id,customer_name,gender_cd,gender,birth_day,age,postal_cd,address,application_store_cd,application_date,status_cd
CS003813000014,村山 菜々美,1,女性,1928-11-26,90,182-0007,東京都調布市菊野台**********,S13003,20160214,0-00000000-0
CS026813000004,吉村 朝陽,1,女性,1928-12-14,90,251-0043,神奈川県藤沢市辻堂元町**********,S14026,20150723,0-00000000-0
CS018811000003,熊沢 美里,1,女性,1929-01-07,90,204-0004,東京都清瀬市野塩**********,S13018,20150403,0-00000000-0
CS027803000004,内村 拓郎,0,男性,1929-01-12,90,251-0031,神奈川県藤沢市鵠沼藤が谷**********,S14027,20151227,0-00000000-0
CS013801000003,天野 拓郎,0,男性,1929-01-15,90,274-0824,千葉県船橋市前原東**********,S12013,20160120,0-00000000-0
CS001814000022,鶴田 里穂,1,女性,1929-01-28,90,144-0045,東京都大田区南六郷**********,S13001,20161012,A-20090415-7
CS016815000002,山元 美紀,1,女性,1929-02-22,90,184-0005,東京都小金井市桜町**********,S13016,20150629,C-20090923-C
CS009815000003,中田 里穂,1,女性,1929-04-08,89,154-0014,東京都世田谷区新町**********,S13009,20150421,D-20091021-E
CS012813000013,宇野 南朋,1,女性,1929-04-09,89,231-0806,神奈川県横浜市中区本牧町**********,S14012,20150712,0-00000000-0
CS005813000015,金谷 恵梨香,1,女性,1929-04-09,89,165-0032,東京都中野区鷺宮**********,S13005,20150506,0-00000000-0


---
> P-18: 顧客データフレーム（df_customer）を生年月日（birth_day）で若い順にソートし、先頭10件を全項目表示せよ。

In [25]:
df_customer.OrderByDescending(r => r.birth_day).Take(10).DisplayTable()

customer_id,customer_name,gender_cd,gender,birth_day,age,postal_cd,address,application_store_cd,application_date,status_cd
CS035114000004,大村 美里,1,女性,2007-11-25,11,156-0053,東京都世田谷区桜**********,S13035,20150619,6-20091205-6
CS022103000002,福山 はじめ,9,不明,2007-10-02,11,249-0006,神奈川県逗子市逗子**********,S14022,20160909,0-00000000-0
CS002113000009,柴田 真悠子,1,女性,2007-09-17,11,184-0014,東京都小金井市貫井南町**********,S13002,20160304,0-00000000-0
CS004115000014,松井 京子,1,女性,2007-08-09,11,165-0031,東京都中野区上鷺宮**********,S13004,20161120,1-20081231-1
CS002114000010,山内 遥,1,女性,2007-06-03,11,184-0015,東京都小金井市貫井北町**********,S13002,20160920,6-20100510-1
CS025115000002,小柳 夏希,1,女性,2007-04-18,11,245-0018,神奈川県横浜市泉区上飯田町**********,S14025,20160116,D-20100913-D
CS002113000025,広末 まなみ,1,女性,2007-03-30,12,184-0015,東京都小金井市貫井北町**********,S13002,20171030,0-00000000-0
CS033112000003,長野 美紀,1,女性,2007-03-22,12,245-0051,神奈川県横浜市戸塚区名瀬町**********,S14033,20150606,0-00000000-0
CS007115000006,福岡 瞬,1,女性,2007-03-10,12,285-0845,千葉県佐倉市西志津**********,S12007,20151118,F-20101016-F
CS014113000008,矢口 莉緒,1,女性,2007-03-05,12,260-0041,千葉県千葉市中央区東千葉**********,S12014,20150622,3-20091108-6


---
> P-19: レシート明細データフレーム（df_receipt）に対し、1件あたりの売上金額（amount）が高い順にランクを付与し、先頭10件を抽出せよ。項目は顧客ID（customer_id）、売上金額（amount）、付与したランクを表示させること。なお、売上金額（amount）が等しい場合は同一順位を付与するものとする。

In [26]:
// 小さい順に並び替えたランキングを計算。同じ値の場合は同じランキングにする
var amount_rank = df_receipt.OrderByDescending(r => r.amount ?? 0)
                .Select((r, index) => new { rank = index + 1, amount = r.amount ?? 0 })
                .GroupBy(r => r.amount, r => r.rank,
                    (amount, rankList) => new { amount = amount, rank = rankList.Min()})
                .ToDictionary(r => r.amount);

// 上記で計算した amount -> ranking の 値を参照する
df_receipt.Select( r => new { r.customer_id, r.amount, ranking = amount_rank[r.amount ?? 0].rank})
    .OrderBy(r => r.ranking).Take(10).DisplayTable()

customer_id,amount,ranking
CS011415000006,10925,1
ZZ000000000000,6800,2
CS028605000002,5780,3
CS015515000034,5480,4
ZZ000000000000,5480,4
ZZ000000000000,5480,4
ZZ000000000000,5440,7
CS021515000089,5440,7
CS015515000083,5280,9
CS017414000114,5280,9


---
> P-020: レシート明細データフレーム（df_receipt）に対し、1件あたりの売上金額（amount）が高い順にランクを付与し、先頭10件を抽出せよ。項目は顧客ID（customer_id）、売上金額（amount）、付与したランクを表示させること。なお、売上金額（amount）が等しい場合でも別順位を付与すること。

In [27]:
df_receipt.OrderByDescending(r => r.amount)
    .Select((r,index) => new { r.customer_id, r.amount, ranking = index + 1})
    .Take(10).DisplayTable()

customer_id,amount,ranking
CS011415000006,10925,1
ZZ000000000000,6800,2
CS028605000002,5780,3
CS015515000034,5480,4
ZZ000000000000,5480,5
ZZ000000000000,5480,6
ZZ000000000000,5440,7
CS021515000089,5440,8
CS015515000083,5280,9
CS017414000114,5280,10


---
> P-021: レシート明細データフレーム（df_receipt）に対し、件数をカウントせよ。

In [28]:
df_receipt.Count

104681

---
> P-022: レシート明細データフレーム（df_receipt）の顧客ID（customer_id）に対し、ユニーク件数をカウントせよ。

In [29]:
df_receipt.Select(r => r.customer_id).Distinct().Count()

8307

---
> P-023: レシート明細データフレーム（df_receipt）に対し、店舗コード（store_cd）ごとに売上金額（amount）と売上数量（quantity）を合計せよ。

In [31]:
//df_receipt.groupby('store_cd').agg({'amount':'sum', 'quantity':'sum'}).reset_index()
df_receipt.GroupBy(r => r.store_cd)
    .Select(r => new {
        store_cd = r.Key,
        sum_amount = r.Sum(t => t.amount ?? 0),
        sum_quantity = r.Sum(t => t.quantity)
    }).OrderBy(r => r.store_cd).DisplayTable()

store_cd,sum_amount,sum_quantity
S12007,638761,2099
S12013,787513,2425
S12014,725167,2358
S12029,794741,2555
S12030,684402,2403
S13001,811936,2347
S13002,727821,2340
S13003,764294,2197
S13004,779373,2390
S13005,629876,2004


---
> P-024: レシート明細データフレーム（df_receipt）に対し、顧客ID（customer_id）ごとに最も新しい売上日（sales_ymd）を求め、10件表示せよ。

In [32]:
df_receipt.GroupBy(r => r.customer_id).Select(r => new {customer_id = r.Key, sales_ymd = r.Max(t => t.sales_ymd)})
    .OrderBy(r => r.customer_id).Take(10).DisplayTable()

customer_id,sales_ymd
CS001113000004,20190308
CS001114000005,20190731
CS001115000010,20190405
CS001205000004,20190625
CS001205000006,20190224
CS001211000025,20190322
CS001212000027,20170127
CS001212000031,20180906
CS001212000046,20170811
CS001212000070,20191018


---
> P-025: レシート明細データフレーム（df_receipt）に対し、顧客ID（customer_id）ごとに最も古い売上日（sales_ymd）を求め、10件表示せよ。

In [33]:
df_receipt.GroupBy(r => r.customer_id).Select(r => new {customer_id = r.Key, sales_ymd = r.Min(t => t.sales_ymd)})
    .OrderBy(r => r.customer_id).Take(10).DisplayTable()

customer_id,sales_ymd
CS001113000004,20190308
CS001114000005,20180503
CS001115000010,20171228
CS001205000004,20170914
CS001205000006,20180207
CS001211000025,20190322
CS001212000027,20170127
CS001212000031,20180906
CS001212000046,20170811
CS001212000070,20191018


---
> P-026: レシート明細データフレーム（df_receipt）に対し、顧客ID（customer_id）ごとに最も新しい売上日（sales_ymd）と古い売上日を求め、両者が異なるデータを10件表示せよ。

In [34]:
df_receipt.GroupBy(r => r.customer_id).Select(r => new {customer_id = r.Key, max_sales_ymd = r.Max(t => t.sales_ymd), min_sales_ymd = r.Min(t => t.sales_ymd) })
    .Where(r => r.max_sales_ymd != r.min_sales_ymd)
    .OrderBy(r => r.customer_id).Take(10).DisplayTable()

customer_id,max_sales_ymd,min_sales_ymd
CS001114000005,20190731,20180503
CS001115000010,20190405,20171228
CS001205000004,20190625,20170914
CS001205000006,20190224,20180207
CS001214000009,20190902,20170306
CS001214000017,20191006,20180828
CS001214000048,20190929,20171109
CS001214000052,20190617,20180208
CS001215000005,20181021,20170206
CS001215000040,20171022,20170214


---
> P-027: レシート明細データフレーム（df_receipt）に対し、店舗コード（store_cd）ごとに売上金額（amount）の平均を計算し、降順でTOP5を表示せよ。

In [35]:
df_receipt.GroupBy(r => r.store_cd).Select(r => new {
    store_cd = r.Key,
    avg_amount = r.Average(r => r.amount)
}).OrderByDescending(r => r.avg_amount).Take(5).DisplayTable()

store_cd,avg_amount
S13052,402.86746987951807
S13015,351.11196043165467
S13003,350.9155188246097
S14010,348.79126213592235
S13001,348.4703862660944


---
> P-028: レシート明細データフレーム（df_receipt）に対し、店舗コード（store_cd）ごとに売上金額（amount）の中央値を計算し、降順でTOP5を表示せよ。

In [36]:
using MathNet.Numerics.Statistics;

df_receipt.GroupBy(r => r.store_cd).Select(r => new {
    store_cd = r.Key,
    median_amount = r.Select(t => (double) t.amount).Median()
}).OrderByDescending(r => r.median_amount).ThenBy(r => r.store_cd).Take(5).DisplayTable()

store_cd,median_amount
S13052,190
S14010,188
S14050,185
S13003,180
S13018,180


---
> P-029: レシート明細データフレーム（df_receipt）に対し、店舗コード（store_cd）ごとに商品コード（product_cd）の最頻値を求めよ。

In [37]:
// store_cd product_cd をキーに データ件数をカウント
var count_list = df_receipt
.GroupBy(r => new { r.store_cd, r.product_cd })
.Select(t => new { store_cd = t.Key.store_cd, product_cd = t.Key.product_cd, count = t.Count() });

// store_cd 単位で 件数の多い最初の1件を抽出
var store_top_count_item = count_list
.GroupBy(r => r.store_cd)
.Select(r => new { store_cd = r.Key, maxItem = r.OrderByDescending(r => r.count).First() });

// 結果取得
store_top_count_item.Select(r => new { r.store_cd, r.maxItem.product_cd, r.maxItem.count }).OrderBy(r => r.store_cd)
.DisplayTable()

store_cd,product_cd,count
S12007,P060303001,72
S12013,P060303001,107
S12014,P060303001,65
S12029,P060303001,92
S12030,P060303001,115
S13001,P060303001,67
S13002,P060303001,78
S13003,P071401001,65
S13004,P060303001,88
S13005,P040503001,36


---
> P-030: レシート明細データフレーム（df_receipt）に対し、店舗コード（store_cd）ごとに売上金額（amount）の標本分散を計算し、降順でTOP5を表示せよ。

In [38]:
using MathNet.Numerics.Statistics;

df_receipt.GroupBy(r => r.store_cd).Select(r => new {
    store_cd = r.Key,
    variance_amount = r.Select(t => (double) t.amount).PopulationVariance()
}).OrderByDescending(r => r.variance_amount).ThenBy(r => r.store_cd).Take(5).DisplayTable()

store_cd,variance_amount
S13052,440088.70131126913
S14011,306314.5581638888
S14034,296920.0810112844
S13001,295431.993329035
S13015,295294.36111594073


---
> P-031: レシート明細データフレーム（df_receipt）に対し、店舗コード（store_cd）ごとに売上金額（amount）の標本標準偏差を計算し、降順でTOP5を表示せよ。

```
var_sample <- function(x){ var(x)*(length(x)-1)/length(x) }
std_sample <- function(x){ sqrt(var_sample(x)) }
```

In [39]:
using MathNet.Numerics.Statistics;

df_receipt.GroupBy(r => r.store_cd)
    .Select(r => new {key = r.Key, std_amount = Math.Sqrt(r.Select(t => (double) t.amount).PopulationVariance())})
    .OrderByDescending(r => r.std_amount)
    .Take(5).DisplayTable()

key,std_amount
S13052,663.3918158307872
S14011,553.456916267101
S14034,544.903735545357
S13001,543.5365611704838
S13015,543.4099383669209


---
> P-032: レシート明細データフレーム（df_receipt）の売上金額（amount）について、25％刻みでパーセンタイル値を求めよ。

In [41]:
using MathNet.Numerics.Statistics;

// 売上金額のリスト
var amountList = df_receipt.Select(r => (double) r.amount).ToList();

// 結果
new { 
    amount_25per = amountList.Percentile(25),
    amount_50per = amountList.Percentile(50),
    amount_75per = amountList.Percentile(75),
    amount_100per = amountList.Percentile(100),
}

amount_25per,102
amount_50per,170
amount_75per,288
amount_100per,10925


---
> P-033: レシート明細データフレーム（df_receipt）に対し、店舗コード（store_cd）ごとに売上金額（amount）の平均を計算し、330以上のものを抽出せよ。

In [42]:
df_receipt.GroupBy(r => r.store_cd)
    .Select(r => new { store_code = r.Key, avg_amount = r.Average(t => t.amount) })
    .Where(r => r.avg_amount >= 330).OrderBy(r => r.store_code).DisplayTable()

store_code,avg_amount
S12013,330.19412997903567
S13001,348.4703862660944
S13003,350.9155188246097
S13004,330.94394904458596
S13015,351.11196043165467
S13019,330.20861587554845
S13020,337.879932117098
S13052,402.86746987951807
S14010,348.79126213592235
S14011,335.7183333333333


---
> P-034: レシート明細データフレーム（df_receipt）に対し、顧客ID（customer_id）ごとに売上金額（amount）を合計して全顧客の平均を求めよ。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。

In [44]:
df_receipt
    .Where(r => !r.customer_id.StartsWith("Z"))
    .GroupBy(r => r.customer_id)
    .Select(r => new { sum_amount = r.Sum(r => r.amount)})
    .Average(r => r.sum_amount)

2547.742234529256

---
> P-035: レシート明細データフレーム（df_receipt）に対し、顧客ID（customer_id）ごとに売上金額（amount）を合計して全顧客の平均を求め、平均以上に買い物をしている顧客を抽出せよ。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。なお、データは10件だけ表示させれば良い。

In [45]:
var cust_amount = df_receipt
    .Where(r => !r.customer_id.StartsWith("Z"))
    .GroupBy(r => r.customer_id)
    .Select(r => new { customer_id = r.Key, sum_amount = r.Sum(r => r.amount)});
var avg_amount = cust_amount.Average(r => r.sum_amount);

cust_amount.Where(r => r.sum_amount >= avg_amount).OrderBy(r => r.customer_id).Take(10).DisplayTable()

customer_id,sum_amount
CS001115000010,3044
CS001205000006,3337
CS001214000009,4685
CS001214000017,4132
CS001214000052,5639
CS001215000040,3496
CS001304000006,3726
CS001305000005,3485
CS001305000011,4370
CS001315000180,3300


---
> P-036: レシート明細データフレーム（df_receipt）と店舗データフレーム（df_store）を内部結合し、レシート明細データフレームの全項目と店舗データフレームの店舗名（store_name）を10件表示させよ。

In [46]:
df_receipt.Join(df_store,
    receipt => receipt.store_cd,
    store => store.store_cd,
    (receipt, store) =>
    new
    {
        receipt.sales_ymd,
        receipt.sales_epoch,
        receipt.store_cd,
        receipt.receipt_no,
        receipt.receipt_sub_no,
        receipt.customer_id,
        receipt.product_cd,
        receipt.quantity,
        receipt.amount,
        store_name = store.store_name,
    }).Take(10).DisplayTable()

sales_ymd,sales_epoch,store_cd,receipt_no,receipt_sub_no,customer_id,product_cd,quantity,amount,store_name
20181103,1541203200,S14006,112,1,CS006214000001,P070305012,1,158,葛が谷店
20181118,1542499200,S13008,1132,2,CS008415000097,P070701017,1,81,成城店
20170712,1499817600,S14028,1102,1,CS028414000014,P060101005,1,170,二ツ橋店
20190205,1549324800,S14042,1132,1,ZZ000000000000,P050301001,1,25,新山下店
20180821,1534809600,S14025,1102,2,CS025415000050,P060102007,1,90,大和店
20190605,1559692800,S13003,1112,1,CS003515000195,P050102002,1,138,狛江店
20181205,1543968000,S14024,1102,2,CS024514000042,P080101005,1,30,三田店
20190922,1569110400,S14040,1102,1,CS040415000178,P070501004,1,128,長津田店
20170504,1493856000,S13020,1112,2,ZZ000000000000,P071302010,1,770,十条仲原店
20191010,1570665600,S14027,1102,1,CS027514000015,P071101003,1,680,南藤沢店


---
> P-037: 商品データフレーム（df_product）とカテゴリデータフレーム（df_category）を内部結合し、商品データフレームの全項目とカテゴリデータフレームの小区分名（category_small_name）を10件表示させよ。

In [47]:
df_product.Join(df_category,
    product => product.category_small_cd,
    category => category.category_small_cd,
    (product, cateogory) => new {
        product.product_cd ,
        product.category_major_cd ,
        product.category_medium_cd ,
        product.category_small_cd ,
        product.unit_price ,
        product.unit_cost ,
        cateogory.category_small_name
    }).Take(10).DisplayTable()

product_cd,category_major_cd,category_medium_cd,category_small_cd,unit_price,unit_cost,category_small_name
P040101001,04,0401,040101,198,149,弁当類
P040101002,04,0401,040101,218,164,弁当類
P040101003,04,0401,040101,230,173,弁当類
P040101004,04,0401,040101,248,186,弁当類
P040101005,04,0401,040101,268,201,弁当類
P040101006,04,0401,040101,298,224,弁当類
P040101007,04,0401,040101,338,254,弁当類
P040101008,04,0401,040101,420,315,弁当類
P040101009,04,0401,040101,498,374,弁当類
P040101010,04,0401,040101,580,435,弁当類


---
> P-038: 顧客データフレーム（df_customer）とレシート明細データフレーム（df_receipt）から、各顧客ごとの売上金額合計を求めよ。ただし、買い物の実績がない顧客については売上金額を0として表示させること。また、顧客は性別コード（gender_cd）が女性（1）であるものを対象とし、非会員（顧客IDが'Z'から始まるもの）は除外すること。なお、結果は10件だけ表示させれば良い。

In [48]:
// 顧客毎の売り上げ金額
var customer_amount_sum = df_receipt.GroupBy(r => r.customer_id).Select(r => new {customer_id = r.Key, sum_amount = r.Sum(t => t.amount ?? 0)})
    .ToDictionary(r => r.customer_id);

// 対象顧客
var customer_id_list = df_customer.Where(r => r.gender_cd == "1" && !r.customer_id.StartsWith("Z"));

// 顧客単位に表示
customer_id_list.Select(r => new {
    r.customer_id,
    sum_amount = customer_amount_sum.GetValueOrDefault(r.customer_id)?.sum_amount ?? 0,
}).Take(10).DisplayTable()

customer_id,sum_amount
CS021313000114,0
CS031415000172,5088
CS028811000001,0
CS001215000145,875
CS015414000103,3122
CS033513000180,868
CS035614000014,0
CS011215000048,3444
CS009413000079,0
CS040412000191,210


---
> P-039: レシート明細データフレーム（df_receipt）から売上日数の多い顧客の上位20件と、売上金額合計の多い顧客の上位20件を抽出し、完全外部結合せよ。ただし、非会員（顧客IDが'Z'から始まるもの）は除外すること。

In [49]:
// 売上金額の多い顧客 TOP 20
var df_sum = df_receipt.GroupBy(r => r.customer_id).Select(r => new { customer_id = r.Key, sum_amount = r.Sum(t => t.amount)}).ToList();
df_sum = df_sum.Where(r => ! r.customer_id.StartsWith("Z")).ToList();
df_sum = df_sum.OrderByDescending(r => r.sum_amount).Take(20).ToList();

// 売上日数の多い顧客
var df_cnt = df_receipt.GroupBy(r => r.customer_id).Select(r => new {
    customer_id = r.Key, 
    sales_ymd_count = r.Select(t => t.sales_ymd).Distinct().Count()});
// 対象外を取り除く
df_cnt = df_cnt.Where(r => !r.customer_id.StartsWith("Z"))
    .OrderByDescending(r => r.sales_ymd_count).Take(20).ToList();
// 顧客一覧
var customer_id_list = df_sum.Select(r => new {r.customer_id} ).Union(df_cnt.Select(r => new {r.customer_id})).Distinct().ToList();

// LEFT JOIN 形式のクエリーを書くのは この形式が書きやすい
var query = from customer_id_list in customer_id_list
        join df_cnt in df_cnt on customer_id_list.customer_id equals df_cnt.customer_id into df_cnt_grouping
        from pp_cnt in df_cnt_grouping.DefaultIfEmpty()
        join df_sum in df_sum on customer_id_list.customer_id equals df_sum.customer_id into df_sum_grouping
        from PP_sum in df_sum_grouping.DefaultIfEmpty()
        select new { 
            customer_id_list.customer_id, 
            PP_sum?.sum_amount, 
            come_days = pp_cnt?.sales_ymd_count,
        };

query.Take(20).ToList().DisplayTable()

customer_id,sum_amount,come_days
CS017415000097,23086,20
CS015415000185,20153,22
CS031414000051,19202,19
CS028415000007,19127,21
CS001605000009,18925,<null>
CS010214000010,18585,22
CS016415000141,18372,20
CS006515000023,18372,<null>
CS011414000106,18338,<null>
CS038415000104,17847,<null>


---
> P-040: 全ての店舗と全ての商品を組み合わせると何件のデータとなるか調査したい。店舗（df_store）と商品（df_product）を直積した件数を計算せよ。

In [50]:
int n_store = df_store.Count();
int n_product = df_product.Count();

n_store * n_product

531590

---
> P-041: レシート明細データフレーム（df_receipt）の売上金額（amount）を日付（sales_ymd）ごとに集計し、前日からの売上金額増減を計算せよ。なお、計算結果は10件表示すればよい。

In [52]:
// 日付毎の売り上げ額
var day_sales_amount = df_receipt.GroupBy(r => r.sales_ymd)
    .Select(r => new {sales_ymd = r.Key, sum_amount = r.Sum(t => t.amount?? 0)})
    .ToDictionary(r => r.sales_ymd);
// 日付計算
string yyyymmdd_add_day(String yyyymmdd, int add_days) {
    DateTime tm = DateTime.Parse(yyyymmdd.Substring(0,4) + "/" + yyyymmdd.Substring(4,2) + "/" + yyyymmdd.Substring(6));
    tm = tm.AddDays(add_days);
    return tm.ToString("yyyyMMdd");
}

var date_list = df_receipt.Select(r => r.sales_ymd).Distinct().OrderBy(r => r).ToList();
date_list.Select(yyyymmdd => new {
    sale_ymd = yyyymmdd,
    amount = day_sales_amount.GetValueOrDefault(yyyymmdd)?.sum_amount ?? 0,
    befor_ymd = yyyymmdd_add_day(yyyymmdd , -1),
}).Select(r => new {
    r.sale_ymd,
    r.amount,
    r.befor_ymd,
    befor_amount = day_sales_amount.GetValueOrDefault(r.befor_ymd)?.sum_amount,
    diff_amount = r.amount - day_sales_amount.GetValueOrDefault(r.befor_ymd)?.sum_amount ?? 0,
}).Take(10).DisplayTable()

sale_ymd,amount,befor_ymd,befor_amount,diff_amount
20170101,33723,20161231,<null>,0
20170102,24165,20170101,33723,-9558
20170103,27503,20170102,24165,3338
20170104,36165,20170103,27503,8662
20170105,37830,20170104,36165,1665
20170106,32387,20170105,37830,-5443
20170107,23415,20170106,32387,-8972
20170108,24737,20170107,23415,1322
20170109,26718,20170108,24737,1981
20170110,20143,20170109,26718,-6575


---
> P-042: レシート明細データフレーム（df_receipt）の売上金額（amount）を日付（sales_ymd）ごとに集計し、各日付のデータに対し、１日前、２日前、３日前のデータを結合せよ。結果は10件表示すればよい。

In [53]:
// 日付毎に集計して 売上金額の 合計を計算
var sales_ymd_to_amount = df_receipt.GroupBy(r => r.sales_ymd).Select(r => new {
    sales_ymd = r.Key,
    amount = r.Sum(t => t.amount),
}).ToDictionary(r => r.sales_ymd); // 日付をキーに検索できるように Dictionary 変換
//display(sales_ymd_to_amount);

// 日付加減算処理
string yyyymmdd_add_day(String yyyymmdd, int add_days) {
    DateTime tm = DateTime.Parse(yyyymmdd.Substring(0,4) + "/" + yyyymmdd.Substring(4,2) + "/" + yyyymmdd.Substring(6));
    tm = tm.AddDays(add_days);
    return tm.ToString("yyyyMMdd");
}

sales_ymd_to_amount.Values
.OrderBy(r => r.sales_ymd)    // データを日付順に並び替え
.Select(r => new {
    // 1日前、2日前、3日前の計算
    sales_ymd = r.sales_ymd,
    amount = r.amount,
    lag_ymd_1 = yyyymmdd_add_day(r.sales_ymd, -1),
    lag_ymd_2 = yyyymmdd_add_day(r.sales_ymd, -2),
    lag_ymd_3 = yyyymmdd_add_day(r.sales_ymd, -3),
}).Select(r => new {
    // 指定の日の 売上金額 設定
    sales_ymd = r.sales_ymd,
    amount = r.amount,
    lag_ymd_1 = r.lag_ymd_1,
    lag_amount_1 = sales_ymd_to_amount.GetValueOrDefault(r.lag_ymd_1)?.amount,
    lag_ymd_2 = r.lag_ymd_2,
    lag_amount_2 = sales_ymd_to_amount.GetValueOrDefault(r.lag_ymd_2)?.amount,
    lag_ymd_3 = r.lag_ymd_3,
    lag_amount_3 = sales_ymd_to_amount.GetValueOrDefault(r.lag_ymd_3)?.amount,
}).DisplayTable()

sales_ymd,amount,lag_ymd_1,lag_amount_1,lag_ymd_2,lag_amount_2,lag_ymd_3,lag_amount_3
20170101,33723,20161231,<null>,20161230,<null>,20161229,<null>
20170102,24165,20170101,33723,20161231,<null>,20161230,<null>
20170103,27503,20170102,24165,20170101,33723,20161231,<null>
20170104,36165,20170103,27503,20170102,24165,20170101,33723
20170105,37830,20170104,36165,20170103,27503,20170102,24165
20170106,32387,20170105,37830,20170104,36165,20170103,27503
20170107,23415,20170106,32387,20170105,37830,20170104,36165
20170108,24737,20170107,23415,20170106,32387,20170105,37830
20170109,26718,20170108,24737,20170107,23415,20170106,32387
20170110,20143,20170109,26718,20170108,24737,20170107,23415


---
> P-043： レシート明細データフレーム（df_receipt）と顧客データフレーム（df_customer）を結合し、性別（gender）と年代（ageから計算）ごとに売上金額（amount）を合計した売上サマリデータフレーム（df_sales_summary）を作成せよ。性別は0が男性、1が女性、9が不明を表すものとする。
>
> ただし、項目構成は年代、女性の売上金額、男性の売上金額、性別不明の売上金額の4項目とすること（縦に年代、横に性別のクロス集計）。また、年代は10歳ごとの階級とすること。

In [55]:
string gender_cd_to_name(string gender_cd) {
    switch(gender_cd) {
        case "0": return "男性";
        case "1": return "女性";
        case "9": return "不明";
        default: return null;
    }
}
var list = from df_receipt in df_receipt
    join df_customer in df_customer on df_receipt.customer_id equals df_customer.customer_id into df_receipt_cusomer
    from df_data in df_receipt_cusomer
    select new {
        df_receipt_customer_id = df_receipt.customer_id,
        df_receipt.amount,
        df_data?.customer_id,
        df_data?.age,
        gender_cd = df_data?.gender_cd,
        era = (df_data?.age ?? 0) / 10 * 10,
    };

var list_sum = list
    .GroupBy(r => r.era) // 世代でグループ化
    .Select(r => new {
        era = r.Key,
        male = r.Where(t => t.gender_cd == "0").Sum(r => r.amount),
        female = r.Where(t => t.gender_cd == "1").Sum(r => r.amount),
        unknown = r.Where(t => t.gender_cd == "9").Sum(r => r.amount),
        null_data = r.Where(t => t.gender_cd == null).Sum(r => r.amount),
    }).OrderBy(r => r.era).ToList();

list_sum.DisplayTable();

era,male,female,unknown,null_data
10,1591,149836,4317,0
20,72940,1363724,44328,0
30,177322,693047,50441,0
40,19355,9320791,483512,0
50,54320,6685192,342923,0
60,272469,987741,71418,0
70,13435,29764,2427,0
80,46360,262923,5111,0
90,0,6260,0,0


---
> P-044： 前設問で作成した売上サマリデータフレーム（df_sales_summary）は性別の売上を横持ちさせたものであった。このデータフレームから性別を縦持ちさせ、年代、性別コード、売上金額の3項目に変換せよ。ただし、性別コードは男性を'00'、女性を'01'、不明を'99'とする。

In [56]:
string gender_cd_convert(string gender_cd) {
        switch(gender_cd) {
            case "0": return "00";
            case "1": return "01";
            case "9": return "99";
            default: return null;
        }
    }

var list = from df_receipt in df_receipt
    join df_customer in df_customer on df_receipt.customer_id equals df_customer.customer_id into df_receipt_cusomer
    from df_data in df_receipt_cusomer
    select new {
        df_receipt.amount,
        gender_cd = gender_cd_convert(df_data?.gender_cd),
        era = (df_data?.age ?? 0) / 10 * 10,
    };

list.GroupBy(r => new { r.era, r.gender_cd})
    .Select(r => new { era = r.Key.era, gender_cd = r.Key.gender_cd, sum_amount = r.Sum(t => t.amount)})
    .OrderBy(r => r.gender_cd).ThenBy(r => r.era).DisplayTable()

era,gender_cd,sum_amount
10,00,1591
20,00,72940
30,00,177322
40,00,19355
50,00,54320
60,00,272469
70,00,13435
80,00,46360
10,01,149836
20,01,1363724


---
> P-045: 顧客データフレーム（df_customer）の生年月日（birth_day）は日付型（Date）でデータを保有している。これをYYYYMMDD形式の文字列に変換し、顧客ID（customer_id）とともに抽出せよ。データは10件を抽出すれば良い。

In [57]:
df_customer.Select(r => new {
    r.customer_id,
    birth_day = r.birth_day?.ToString("yyyyMMdd"),
}).Take(10).DisplayTable()

customer_id,birth_day
CS021313000114,19810429
CS037613000071,19520401
CS031415000172,19761004
CS028811000001,19330327
CS001215000145,19950329
CS020401000016,19740915
CS015414000103,19770809
CS029403000008,19730817
CS015804000004,19310502
CS033513000180,19620711


---
> P-046: 顧客データフレーム（df_customer）の申し込み日（application_date）はYYYYMMDD形式の文字列型でデータを保有している。これを日付型（dateやdatetime）に変換し、顧客ID（customer_id）とともに抽出せよ。データは10件を抽出すれば良い。

In [58]:

DateTime ?yyyymmdd_to_date(string yyyymmdd) {
    if (yyyymmdd == null) return null;
    return DateTime.Parse(yyyymmdd.Substring(0,4) + "/" + yyyymmdd.Substring(4,2) + "/" + yyyymmdd.Substring(4,2));
}
df_customer.Select(r => new {
    r.customer_id,
    application_date = yyyymmdd_to_date(r.application_date),
}).Take(5).DisplayTable()

customer_id,application_date
CS021313000114,2015-09-09
CS037613000071,2015-04-04
CS031415000172,2015-05-05
CS028811000001,2016-01-01
CS001215000145,2017-06-06


---
> P-047: レシート明細データフレーム（df_receipt）の売上日（sales_ymd）はYYYYMMDD形式の数値型でデータを保有している。これを日付型（dateやdatetime）に変換し、レシート番号(receipt_no)、レシートサブ番号（receipt_sub_no）とともに抽出せよ。データは10件を抽出すれば良い。

In [59]:
DateTime ?yyyymmdd_to_date(string yyyymmdd) {
    if (yyyymmdd == null) return null;
    return DateTime.Parse(yyyymmdd.Substring(0,4) + "/" + yyyymmdd.Substring(4,2) + "/" + yyyymmdd.Substring(6));
}

df_receipt.Select(r => new {
    r.receipt_no,
    r.receipt_sub_no,
    sales_ymd = yyyymmdd_to_date(r.sales_ymd),
}).Take(10).DisplayTable()

receipt_no,receipt_sub_no,sales_ymd
112,1,2018-11-03
1132,2,2018-11-18
1102,1,2017-07-12
1132,1,2019-02-05
1102,2,2018-08-21
1112,1,2019-06-05
1102,2,2018-12-05
1102,1,2019-09-22
1112,2,2017-05-04
1102,1,2019-10-10


---
> P-048: レシート明細データフレーム（df_receipt）の売上エポック秒（sales_epoch）は数値型のUNIX秒でデータを保有している。これを日付型（dateやdatetime）に変換し、レシート番号(receipt_no)、レシートサブ番号（receipt_sub_no）とともに抽出せよ。データは10件を抽出すれば良い。

In [60]:
// df_receipt.Where(r => r.sales_epoch == null).Count()
// null は 0 件
DateTime epoch_to_date(int? epoch) {
    return DateTime.UnixEpoch.AddSeconds(epoch ?? 0);
}

df_receipt.Select(r => new {
    r.receipt_no,
    r.receipt_sub_no,
    sales_ymd = epoch_to_date(r.sales_epoch),
}).Take(10).DisplayTable()

receipt_no,receipt_sub_no,sales_ymd
112,1,2018-11-03
1132,2,2018-11-18
1102,1,2017-07-12
1132,1,2019-02-05
1102,2,2018-08-21
1112,1,2019-06-05
1102,2,2018-12-05
1102,1,2019-09-22
1112,2,2017-05-04
1102,1,2019-10-10


---
> P-049: レシート明細データフレーム（df_receipt）の売上エポック秒（sales_epoch）を日付型（timestamp型）に変換し、"年"だけ取り出してレシート番号(receipt_no)、レシートサブ番号（receipt_sub_no）とともに抽出せよ。データは10件を抽出すれば良い。

In [61]:
df_receipt.Select(r => new {
    r.receipt_no,
    r.receipt_sub_no,
    sales_year = epoch_to_date(r.sales_epoch).Year,
}).Take(10).DisplayTable()

receipt_no,receipt_sub_no,sales_year
112,1,2018
1132,2,2018
1102,1,2017
1132,1,2019
1102,2,2018
1112,1,2019
1102,2,2018
1102,1,2019
1112,2,2017
1102,1,2019


---
> P-050: レシート明細データフレーム（df_receipt）の売上エポック秒（sales_epoch）を日付型（timestamp型）に変換し、"月"だけ取り出してレシート番号(receipt_no)、レシートサブ番号（receipt_sub_no）とともに抽出せよ。なお、"月"は0埋め2桁で取り出すこと。データは10件を抽出すれば良い。

In [62]:
df_receipt.Select(r => new {
    r.receipt_no,
    r.receipt_sub_no,
    sales_month = epoch_to_date(r.sales_epoch).Month.ToString("D2"),
}).Take(10).DisplayTable()

receipt_no,receipt_sub_no,sales_month
112,1,11
1132,2,11
1102,1,07
1132,1,02
1102,2,08
1112,1,06
1102,2,12
1102,1,09
1112,2,05
1102,1,10


---
> P-051: レシート明細データフレーム（df_receipt）の売上エポック秒（sales_epoch）を日付型（timestamp型）に変換し、"日"だけ取り出してレシート番号(receipt_no)、レシートサブ番号（receipt_sub_no）とともに抽出せよ。なお、"日"は0埋め2桁で取り出すこと。データは10件を抽出すれば良い。

In [63]:
df_receipt.Select(r => new {
    r.receipt_no,
    r.receipt_sub_no,
    sales_month = epoch_to_date(r.sales_epoch).Day.ToString("D2"),
}).Take(10).DisplayTable()

receipt_no,receipt_sub_no,sales_month
112,1,03
1132,2,18
1102,1,12
1132,1,05
1102,2,21
1112,1,05
1102,2,05
1102,1,22
1112,2,04
1102,1,10


---
> P-052: レシート明細データフレーム（df_receipt）の売上金額（amount）を顧客ID（customer_id）ごとに合計の上、売上金額合計に対して2000円以下を0、2000円超を1に2値化し、顧客ID、売上金額合計とともに10件表示せよ。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。

In [64]:
df_receipt
    .Where(r => !r.customer_id.StartsWith("Z")) // 顧客ID が Z から始まるデータは除外
    .GroupBy(r => r.customer_id)
    .Select(r => new {
        cusomer_id = r.Key,
        sum_amount = r.Sum(t => t.amount)
    })
    .Select(r => new {
        r.cusomer_id,
        r.sum_amount,
        bit_amount = r.sum_amount <= 2000 ? 0 : 1,
    })
    .OrderBy(r => r.cusomer_id).Take(10).DisplayTable()

cusomer_id,sum_amount,bit_amount
CS001113000004,1298,0
CS001114000005,626,0
CS001115000010,3044,1
CS001205000004,1988,0
CS001205000006,3337,1
CS001211000025,456,0
CS001212000027,448,0
CS001212000031,296,0
CS001212000046,228,0
CS001212000070,456,0


---
> P-053: 顧客データフレーム（df_customer）の郵便番号（postal_cd）に対し、東京（先頭3桁が100〜209のもの）を1、それ以外のものを0に２値化せよ。さらにレシート明細データフレーム（df_receipt）と結合し、全期間において買い物実績のある顧客数を、作成した2値ごとにカウントせよ。

In [65]:
// 郵便番号先頭3桁を 分類
string bit_postal_cd(string postal_cd) {
    if (postal_cd.Length < 3) return "0";
    string postal_3 = postal_cd.Substring(0,3);
    int posCode;
    if (int.TryParse(postal_3, out posCode)) {
        if (100 <= posCode && posCode <= 209) {
            return "1";
        }
    }
    return "0";
}

// カスタマーID 毎に 住所コードからフラグ判定
var postal_flg_dic = df_customer.Select(r => new {
    postal_flg = bit_postal_cd(r.postal_cd),
    r.customer_id,
}).ToDictionary(r => r.customer_id);

df_receipt.Select(r => r.customer_id).Distinct()
    .Select(cusomer_id => new {
        cusomer_id = cusomer_id,
        postal_flg = postal_flg_dic.GetValueOrDefault(cusomer_id)?.postal_flg,
    })
    .GroupBy(r => r.postal_flg)
    .Select(r => new {
        postal_flg = r.Key,
        sum_customer_id = r.Count()
    }).DisplayTable()

postal_flg,sum_customer_id
0,3906
1,4400
<null>,1


---
> P-054: 顧客データデータフレーム（df_customer）の住所（address）は、埼玉県、千葉県、東京都、神奈川県のいずれかとなっている。都道府県毎にコード値を作成し、顧客ID、住所とともに抽出せよ。値は埼玉県を11、千葉県を12、東京都を13、神奈川県を14とすること。結果は10件表示させれば良い。

In [66]:
//pd.concat([df_customer[['customer_id', 'address']], df_customer['address'].str[0:3].map({'埼玉県': '11',
//                                                                           '千葉県':'12', 
//                                                                           '東京都':'13', 
//                                                                           '神奈川':'14'})], axis=1).head(10)
string addressToCode(string address) {
    if (address.StartsWith("埼玉県")) return "11";
    if (address.StartsWith("千葉県")) return "12";
    if (address.StartsWith("東京都")) return "13";
    if (address.StartsWith("神奈川県")) return "14";
    else return "";
}

df_customer.Select(r => new {
    r.customer_id,
    r.address,
    address_code = addressToCode(r.address),
}).Take(10).DisplayTable()

customer_id,address,address_code
CS021313000114,神奈川県伊勢原市粟窪**********,14
CS037613000071,東京都江東区南砂**********,13
CS031415000172,東京都渋谷区代々木**********,13
CS028811000001,神奈川県横浜市泉区和泉町**********,14
CS001215000145,東京都大田区仲六郷**********,13
CS020401000016,東京都板橋区若木**********,13
CS015414000103,東京都江東区北砂**********,13
CS029403000008,千葉県浦安市海楽**********,12
CS015804000004,東京都江東区北砂**********,13
CS033513000180,神奈川県横浜市旭区善部町**********,14


---
> P-055: レシート明細データフレーム（df_receipt）の売上金額（amount）を顧客ID（customer_id）ごとに合計し、その合計金額の四分位点を求めよ。その上で、顧客ごとの売上金額合計に対して以下の基準でカテゴリ値を作成し、顧客ID、売上金額と合計ともに表示せよ。カテゴリ値は上から順に1〜4とする。結果は10件表示させれば良い。
>
> - 最小値以上第一四分位未満
> - 第一四分位以上第二四分位未満
> - 第二四分位以上第三四分位未満
> - 第三四分位以上

In [67]:
using MathNet.Numerics.Statistics;

// 顧客ID 毎の 売上金額合計
var df_sum = df_receipt.GroupBy(r => r.customer_id)
    .Select(r => new { customer_id = r.Key, sum_amount = (double) r.Sum(t => t.amount)});

// 分位点を求める
var pct = new {
    pct25 = Statistics.Quantile(df_sum.Select(r => r.sum_amount), 0.25),
    pct50 = Statistics.Quantile(df_sum.Select(r => r.sum_amount), 0.5),
    pct75 = Statistics.Quantile(df_sum.Select(r => r.sum_amount), 0.75),
};
display(pct);
// 分位点のカテゴリ計算
int amountCategory(double amount) {
    if (amount < pct.pct25) return 1;
    if (amount < pct.pct50) return 2;
    if (amount < pct.pct75) return 3;
    return 4;
}
// 集計
df_sum.Select(r => new {
    r.customer_id,
    r.sum_amount,
    amount_category = amountCategory(r.sum_amount),
}).OrderBy(r => r.customer_id).Take(10).DisplayTable()

pct25,548.166666666667
pct50,1478
pct75,3651.666666666666


customer_id,sum_amount,amount_category
CS001113000004,1298,2
CS001114000005,626,2
CS001115000010,3044,3
CS001205000004,1988,3
CS001205000006,3337,3
CS001211000025,456,1
CS001212000027,448,1
CS001212000031,296,1
CS001212000046,228,1
CS001212000070,456,1


---
> P-056: 顧客データフレーム（df_customer）の年齢（age）をもとに10歳刻みで年代を算出し、顧客ID（customer_id）、生年月日（birth_day）とともに抽出せよ。ただし、60歳以上は全て60歳代とすること。年代を表すカテゴリ名は任意とする。先頭10件を表示させればよい。

In [68]:
int? ageCategory(int? age) {
    if (age == null) return null;
    if (age >= 60) return 60;
    return ((int) age / 10) * 10;
}

df_customer.Select(r => new {
    r.customer_id,
    r.birth_day,
    era = ageCategory(r.age),
}).Take(10).DisplayTable()

customer_id,birth_day,era
CS021313000114,1981-04-29,30
CS037613000071,1952-04-01,60
CS031415000172,1976-10-04,40
CS028811000001,1933-03-27,60
CS001215000145,1995-03-29,20
CS020401000016,1974-09-15,40
CS015414000103,1977-08-09,40
CS029403000008,1973-08-17,40
CS015804000004,1931-05-02,60
CS033513000180,1962-07-11,50


---
> P-057: 前問題の抽出結果と性別（gender）を組み合わせ、新たに性別×年代の組み合わせを表すカテゴリデータを作成せよ。組み合わせを表すカテゴリの値は任意とする。先頭10件を表示させればよい。

In [69]:
int? ageCategory(int? age) {
    if (age == null) return null;
    if (age >= 60) return 60;
    return ((int) age / 10) * 10;
}

df_customer.Select(r => new {
    r.customer_id,
    r.birth_day,
    r.gender_cd,
    age = ageCategory(r.age),
}).Select(r => new {
    r.customer_id,
    r.birth_day,
    r.age,
    gender_era_cd = r.gender_cd + "_" + r.age ,
}).Take(10).DisplayTable()

customer_id,birth_day,age,gender_era_cd
CS021313000114,1981-04-29,30,1_30
CS037613000071,1952-04-01,60,9_60
CS031415000172,1976-10-04,40,1_40
CS028811000001,1933-03-27,60,1_60
CS001215000145,1995-03-29,20,1_20
CS020401000016,1974-09-15,40,0_40
CS015414000103,1977-08-09,40,1_40
CS029403000008,1973-08-17,40,0_40
CS015804000004,1931-05-02,60,0_60
CS033513000180,1962-07-11,50,1_50


---
> P-058: 顧客データフレーム（df_customer）の性別コード（gender_cd）をダミー変数化し、顧客ID（customer_id）とともに抽出せよ。結果は10件表示させれば良い。

In [70]:
df_customer.Select(r => new {
    r.customer_id,
    gender_cd_0 = r.gender_cd == "0" ? "1" : "0",
    gender_cd_1 = r.gender_cd == "1" ? "1" : "0",
    gender_cd_9 = r.gender_cd == "9" ? "1" : "0",
}).Take(10).DisplayTable()

customer_id,gender_cd_0,gender_cd_1,gender_cd_9
CS021313000114,0,1,0
CS037613000071,0,0,1
CS031415000172,0,1,0
CS028811000001,0,1,0
CS001215000145,0,1,0
CS020401000016,1,0,0
CS015414000103,0,1,0
CS029403000008,1,0,0
CS015804000004,1,0,0
CS033513000180,0,1,0


---
> P-059: レシート明細データフレーム（df_receipt）の売上金額（amount）を顧客ID（customer_id）ごとに合計し、合計した売上金額を平均0、標準偏差1に標準化して顧客ID、売上金額合計とともに表示せよ。標準化に使用する標準偏差は、不偏標準偏差と標本標準偏差のどちらでも良いものとする。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。結果は10件表示させれば良い。

In [71]:
using MathNet.Numerics.Statistics;

var df_sum = df_receipt.Where(r => ! r.customer_id.StartsWith("Z"))
    .GroupBy(r => r.customer_id)
    .Select(r => new {
        customer_id = r.Key,
        sum_amount = r.Sum(t => t.amount),
    }).ToList();

// 平均と 標準偏差を求める
var def = new {
    avg = df_sum.Select(r => (double) r.sum_amount).Average(),
    st_dev = df_sum.Select(r => (double) r.sum_amount).StandardDeviation(),
};

var df_sum2 = df_sum.Select(r => new {
    r.customer_id,
    // 平均0、標準偏差1に標準化するために 計算
    normalized_amount = (r.sum_amount - def.avg ) / def.st_dev 
});

df_sum2.OrderBy(r => r.customer_id).Take(10).DisplayTable()


customer_id,normalized_amount
CS001113000004,-0.4593502260498646
CS001114000005,-0.7063478415395753
CS001115000010,0.18240250705286576
CS001205000004,-0.20573660300239383
CS001205000006,0.29009640785418306
CS001211000025,-0.7688323573628651
CS001212000027,-0.7717728051663141
CS001212000031,-0.8276413134318439
CS001212000046,-0.8526351197611598
CS001212000070,-0.7688323573628651


---
> P-060: レシート明細データフレーム（df_receipt）の売上金額（amount）を顧客ID（customer_id）ごとに合計し、合計した売上金額を最小値0、最大値1に正規化して顧客ID、売上金額合計とともに表示せよ。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。結果は10件表示させれば良い。

In [72]:
using MathNet.Numerics.Statistics;

var df_sum = df_receipt.Where(r => ! r.customer_id.StartsWith("Z"))
    .GroupBy(r => r.customer_id)
    .Select(r => new {
        customer_id = r.Key,
        sum_amount = r.Sum(t => t.amount),
    }).ToList();

// 最大と最小を求める
var def = new {
    max = df_sum.Select(r => (double) r.sum_amount).Max(),
    min = df_sum.Select(r => (double) r.sum_amount).Min(),
};

var df_sum2 = df_sum.Select(r => new {
    r.customer_id,
    // 最小値0、最大値1に正規化
    normalized_amount = (r.sum_amount - def.min ) / ( def.max - def.min) ,
});

df_sum2.OrderBy(r => r.customer_id).Take(10).DisplayTable()

customer_id,normalized_amount
CS001113000004,0.05335418839068474
CS001114000005,0.02415710809871394
CS001115000010,0.12921445950643032
CS001205000004,0.08333333333333333
CS001205000006,0.14194473409801878
CS001211000025,0.016770941953423706
CS001212000027,0.016423357664233577
CS001212000031,0.009819256169621133
CS001212000046,0.00686478971150504
CS001212000070,0.016770941953423706


---
> P-061: レシート明細データフレーム（df_receipt）の売上金額（amount）を顧客ID（customer_id）ごとに合計し、合計した売上金額を常用対数化（底=10）して顧客ID、売上金額合計とともに表示せよ。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。結果は10件表示させれば良い。

In [73]:
df_receipt
    .Where(r => !r.customer_id.StartsWith("Z")) // 対象外を除外
    .GroupBy(r => r.customer_id) // 顧客ID で集計
    .Select(r => new {
        customer_id = r.Key,
        sum_amount = r.Sum(t => t.amount) // 売上金額を合計
    })
    .Select(r => new {
        r.customer_id,
        r.sum_amount,
        amount_log10 = Math.Log10((double)r.sum_amount), // Log （底 = 10) の計算
    }).OrderBy(r => r.customer_id).Take(10).DisplayTable() // 結果を10件表示

customer_id,sum_amount,amount_log10
CS001113000004,1298,3.1132746924643504
CS001114000005,626,2.7965743332104296
CS001115000010,3044,3.4834446480985353
CS001205000004,1988,3.2984163800612945
CS001205000006,3337,3.5233562066547925
CS001211000025,456,2.658964842664435
CS001212000027,448,2.651278013998144
CS001212000031,296,2.4712917110589387
CS001212000046,228,2.357934847000454
CS001212000070,456,2.658964842664435


---
> P-062: レシート明細データフレーム（df_receipt）の売上金額（amount）を顧客ID（customer_id）ごとに合計し、合計した売上金額を自然対数化(底=e）して顧客ID、売上金額合計とともに表示せよ。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。結果は10件表示させれば良い。

In [74]:
df_receipt
    .Where(r => !r.customer_id.StartsWith("Z")) // 対象外を除外
    .GroupBy(r => r.customer_id) // 顧客ID で集計
    .Select(r => new {
        customer_id = r.Key,
        sum_amount = r.Sum(t => t.amount) // 売上金額を合計
    })
    .Select(r => new {
        r.customer_id,
        r.sum_amount,
        amount_log10 = Math.Log((double)r.sum_amount), // Log （底 = e) の計算
    }).OrderBy(r => r.customer_id).Take(10).DisplayTable() // 結果を10件表示

    // 回答例では 対数計算の時に +1 していた。 Log(0) は マイナス無限大になることの回避だと思うが・・ あまり良いアイデアだとは思わない。

customer_id,sum_amount,amount_log10
CS001113000004,1298,7.168579897264035
CS001114000005,626,6.439350371100098
CS001115000010,3044,8.020927718981577
CS001205000004,1988,7.59488438721652
CS001205000006,3337,8.112827478751374
CS001211000025,456,6.1224928095143865
CS001212000027,448,6.104793232414985
CS001212000031,296,5.69035945432406
CS001212000046,228,5.429345628954441
CS001212000070,456,6.1224928095143865


---
> P-063: 商品データフレーム（df_product）の単価（unit_price）と原価（unit_cost）から、各商品の利益額を算出せよ。結果は10件表示させれば良い。

In [75]:
df_product.Select(r => new {
    r.product_cd,
    r.category_major_cd,
    r.category_medium_cd,
    r.category_small_cd,
    r.unit_price,
    r.unit_cost,
    unit_profit = r.unit_price - r.unit_cost,
}).Take(10).DisplayTable()

product_cd,category_major_cd,category_medium_cd,category_small_cd,unit_price,unit_cost,unit_profit
P040101001,04,0401,040101,198,149,49
P040101002,04,0401,040101,218,164,54
P040101003,04,0401,040101,230,173,57
P040101004,04,0401,040101,248,186,62
P040101005,04,0401,040101,268,201,67
P040101006,04,0401,040101,298,224,74
P040101007,04,0401,040101,338,254,84
P040101008,04,0401,040101,420,315,105
P040101009,04,0401,040101,498,374,124
P040101010,04,0401,040101,580,435,145


---
> P-064: 商品データフレーム（df_product）の単価（unit_price）と原価（unit_cost）から、各商品の利益率の全体平均を算出せよ。
ただし、単価と原価にはNULLが存在することに注意せよ。

In [76]:
df_product
.Where(r => r.unit_cost != null && r.unit_price != null)
.Select(r => new {
    r.product_cd,
    unit_profit_rate = (double) (r.unit_price - r.unit_cost) / r.unit_price,
})
.Average(r => r.unit_profit_rate)

0.24911389885176904

---
> P-065: 商品データフレーム（df_product）の各商品について、利益率が30%となる新たな単価を求めよ。ただし、1円未満は切り捨てること。そして結果を10件表示させ、利益率がおよそ30％付近であることを確認せよ。ただし、単価（unit_price）と原価（unit_cost）にはNULLが存在することに注意せよ。

In [77]:
var new_df = 
df_product
.Where(r => r.unit_cost != null)
.Select(r => new {
    r.product_cd,
    r.unit_cost,
    new_price = Math.Floor((r.unit_cost ?? 0) / 0.7) // 小数点以下切り捨て
});

new_df
.Select(r => new {
    r.product_cd,
    r.new_price,
    r.unit_cost,
    unit_profit_rate = (double) (r.new_price - r.unit_cost) / r.new_price,
}).Take(10).DisplayTable()

product_cd,new_price,unit_cost,unit_profit_rate
P040101001,212,149,0.2971698113207547
P040101002,234,164,0.29914529914529914
P040101003,247,173,0.29959514170040485
P040101004,265,186,0.2981132075471698
P040101005,287,201,0.29965156794425085
P040101006,320,224,0.3
P040101007,362,254,0.2983425414364641
P040101008,450,315,0.3
P040101009,534,374,0.299625468164794
P040101010,621,435,0.2995169082125604


---
> P-066: 商品データフレーム（df_product）の各商品について、利益率が30%となる新たな単価を求めよ。今回は、1円未満を四捨五入すること（0.5については偶数方向の丸めで良い）。そして結果を10件表示させ、利益率がおよそ30％付近であることを確認せよ。ただし、単価（unit_price）と原価（unit_cost）にはNULLが存在することに注意せよ。

In [78]:
var new_df = 
df_product
.Where(r => r.unit_cost != null)
.Select(r => new {
    r.product_cd,
    r.unit_cost,
    new_price = Math.Round((r.unit_cost ?? 0) / 0.7 ) // 1円未満を四捨五入 ( 明示的に指定しない場合は 偶数丸め )
    // https://docs.microsoft.com/ja-jp/dotnet/api/system.math.round?view=net-5.0#System_Math_Round_System_Decimal_
});

new_df
.Select(r => new {
    r.product_cd,
    r.new_price,
    r.unit_cost,
    unit_profit_rate = (double) (r.new_price - r.unit_cost) / r.new_price,
}).Take(10).DisplayTable()

product_cd,new_price,unit_cost,unit_profit_rate
P040101001,213,149,0.3004694835680751
P040101002,234,164,0.29914529914529914
P040101003,247,173,0.29959514170040485
P040101004,266,186,0.3007518796992481
P040101005,287,201,0.29965156794425085
P040101006,320,224,0.3
P040101007,363,254,0.3002754820936639
P040101008,450,315,0.3
P040101009,534,374,0.299625468164794
P040101010,621,435,0.2995169082125604


---
> P-067: 商品データフレーム（df_product）の各商品について、利益率が30%となる新たな単価を求めよ。今回は、1円未満を切り上げること。そして結果を10件表示させ、利益率がおよそ30％付近であることを確認せよ。ただし、単価（unit_price）と原価（unit_cost）にはNULLが存在することに注意せよ。

In [79]:
var new_df = 
df_product
.Where(r => r.unit_cost != null)
.Select(r => new {
    r.product_cd,
    r.unit_cost,
    new_price = Math.Ceiling((r.unit_cost ?? 0) / 0.7 ) // 1円未満を切り上げ( 正の大きい方へ 丸め )
    // https://docs.microsoft.com/ja-jp/dotnet/api/system.math.ceiling?view=net-5.0#System_Math_Ceiling_System_Double_
});

new_df
.Select(r => new {
    r.product_cd,
    r.new_price,
    r.unit_cost,
    unit_profit_rate = (double) (r.new_price - r.unit_cost) / r.new_price,
}).Take(10).DisplayTable()

product_cd,new_price,unit_cost,unit_profit_rate
P040101001,213,149,0.3004694835680751
P040101002,235,164,0.3021276595744681
P040101003,248,173,0.3024193548387097
P040101004,266,186,0.3007518796992481
P040101005,288,201,0.3020833333333333
P040101006,320,224,0.3
P040101007,363,254,0.3002754820936639
P040101008,451,315,0.30155210643015523
P040101009,535,374,0.30093457943925234
P040101010,622,435,0.30064308681672025


---
> P-068: 商品データフレーム（df_product）の各商品について、消費税率10%の税込み金額を求めよ。 1円未満の端数は切り捨てとし、結果は10件表示すれば良い。ただし、単価（unit_price）にはNULLが存在することに注意せよ。

In [80]:
df_product.Select(r => new {
    r.product_cd,
    r.unit_price,
    price_tax = ( r.unit_price != null ? Math.Floor(r.unit_price.Value * 1.1 ) : (double?) null) 
}).Take(10).DisplayTable()

product_cd,unit_price,price_tax
P040101001,198,217
P040101002,218,239
P040101003,230,253
P040101004,248,272
P040101005,268,294
P040101006,298,327
P040101007,338,371
P040101008,420,462
P040101009,498,547
P040101010,580,638


---
> P-069: レシート明細データフレーム（df_receipt）と商品データフレーム（df_product）を結合し、顧客毎に全商品の売上金額合計と、カテゴリ大区分（category_major_cd）が"07"（瓶詰缶詰）の売上金額合計を計算の上、両者の比率を求めよ。抽出対象はカテゴリ大区分"07"（瓶詰缶詰）の購入実績がある顧客のみとし、結果は10件表示させればよい。

In [81]:
// 顧客毎の全商品の売上金額合計
var df_cust_amount = df_receipt.GroupBy(r => r.customer_id)
    .Select(r => new {
        customer_id = r.Key,
        sum_amount = r.Sum(t => t.amount)
    }).ToDictionary(r => r.customer_id);

// 顧客毎の カテゴリ大分類７ の 売上金額合計
var df_cat7_receipt = from df_receipt in df_receipt 
    join df_product in df_product on df_receipt.product_cd equals df_product.product_cd
    where df_product.category_major_cd == "07"
    select new {
        df_receipt.customer_id,
        df_receipt.amount
    };
var df_cat7_amount = df_cat7_receipt.GroupBy(r => r.customer_id)
    .Select(r => new {
        customer_id = r.Key,
        sum_amount = r.Sum(t => t.amount)
    }).ToDictionary(r => r.customer_id);

// 抽出対象顧客一覧
var custno_list = df_cat7_amount.Keys.OrderBy(r => r).ToList();

// 結果出力
custno_list.Select(customer_id => new {
    customer_id = customer_id,
    amount_all = df_cust_amount.GetValueOrDefault(customer_id)?.sum_amount,
    amount_cat_7 = df_cat7_amount.GetValueOrDefault(customer_id)?.sum_amount,
}).Select(r => new {
    r.customer_id,
    r.amount_all,
    r.amount_cat_7,
    amount_rate = (double) r.amount_cat_7 / r.amount_all,
}).Take(10).DisplayTable()

customer_id,amount_all,amount_cat_7,amount_rate
CS001113000004,1298,1298,1
CS001114000005,626,486,0.7763578274760383
CS001115000010,3044,2694,0.8850197109067017
CS001205000004,1988,346,0.17404426559356137
CS001205000006,3337,2004,0.600539406652682
CS001212000027,448,200,0.44642857142857145
CS001212000031,296,296,1
CS001212000046,228,108,0.47368421052631576
CS001212000070,456,308,0.6754385964912281
CS001213000018,243,145,0.5967078189300411


---
> P-070: レシート明細データフレーム（df_receipt）の売上日（sales_ymd）に対し、顧客データフレーム（df_customer）の会員申込日（application_date）からの経過日数を計算し、顧客ID（customer_id）、売上日、会員申込日とともに表示せよ。結果は10件表示させれば良い（なお、sales_ymdは数値、application_dateは文字列でデータを保持している点に注意）。

In [82]:
// YYYYMMDD 形式の日付を DateTime 型に変換する
DateTime ?YYYYMMDDToDateTime(string yyyymmdd) {
    if (yyyymmdd != null && yyyymmdd.Length == 8) {
        return new DateTime(
            int.Parse(yyyymmdd.Substring(0,4)),
            int.Parse(yyyymmdd.Substring(4,2)),
            int.Parse(yyyymmdd.Substring(6)));
    } else {
        return null;
    }
}
// ２つの日付の差の日を計算
int? DiffDate(string yyyymmdd1 , string yyyymmdd2) {
    DateTime ?t1 = YYYYMMDDToDateTime(yyyymmdd1);
    DateTime ?t2 = YYYYMMDDToDateTime(yyyymmdd2);
    if (t1 != null && t2 != null) {
        int diffDay = (int) t2.Value.Subtract(t1.Value).TotalDays;
        return Math.Abs(diffDay);
    } else {
        return null;
    }
}

// レシート明細 join 顧客 
var query = from df_receipt in df_receipt
    join df_customer in df_customer on df_receipt.customer_id equals df_customer.customer_id
    select new {
        df_receipt.customer_id,
        df_receipt.sales_ymd,
        df_customer.application_date,
        // 経過した日を求める
        elapsed_date = DiffDate(df_receipt.sales_ymd, df_customer.application_date),
    };

// 結果の表示
query.Take(10).DisplayTable()

customer_id,sales_ymd,application_date,elapsed_date
CS006214000001,20181103,20150201,1371
CS008415000097,20181118,20150322,1337
CS028414000014,20170712,20150711,732
CS025415000050,20180821,20160131,933
CS003515000195,20190605,20150306,1552
CS024514000042,20181205,20151010,1152
CS040415000178,20190922,20150627,1548
CS027514000015,20191010,20151101,1439
CS025415000134,20190918,20150720,1521
CS021515000126,20171010,20150508,886


---
> P-071: レシート明細データフレーム（df_receipt）の売上日（sales_ymd）に対し、顧客データフレーム（df_customer）の会員申込日（application_date）からの経過月数を計算し、顧客ID（customer_id）、売上日、会員申込日とともに表示せよ。結果は10件表示させれば良い（なお、sales_ymdは数値、application_dateは文字列でデータを保持している点に注意）。1ヶ月未満は切り捨てること。

In [83]:
DateTime ?YYYYMMDDToDateTime(string yyyymmdd) {
    if (yyyymmdd != null && yyyymmdd.Length == 8) {
        return new DateTime(
            int.Parse(yyyymmdd.Substring(0,4)),
            int.Parse(yyyymmdd.Substring(4,2)),
            int.Parse(yyyymmdd.Substring(6)));
    } else {
        return null;
    }
}
// ２つの日付の差の月を計算
int? DiffMonth(string yyyymmdd1 , string yyyymmdd2) {
    DateTime ?t1 = YYYYMMDDToDateTime(yyyymmdd1);
    DateTime ?t2 = YYYYMMDDToDateTime(yyyymmdd2);
    if (t1 != null && t2 != null) {
        if (t1.Value > t2.Value) {
            (t1,t2) = (t2,t1); // 値の入れ替え
        }
        int diff_month = 0;
#if false
        // 29日30日31日 の扱いを工夫
        while (t1.Value.AddMonths(1) <= t2.Value) {
            t1 = t1.Value.AddMonths(1);
            diff_month++;
        }
#else
        // ２つの日付の月の差
        int month1 = t1.Value.Year * 12 + t1.Value.Month;
        int month2 = t2.Value.Year * 12 + t2.Value.Month;
        diff_month = month2 - month1;
        // 日数の大小関係を見て 月を減らす
        if (t1.Value.Day > t2.Value.Day) {
            diff_month --;
        }
#endif
        return diff_month;
    } else {
        return null;
    }
}

// レシート明細 join 顧客 
var query = from df_receipt in df_receipt
    join df_customer in df_customer on df_receipt.customer_id equals df_customer.customer_id
    select new {
        df_receipt.customer_id,
        df_receipt.sales_ymd,
        df_customer.application_date,
        // 経過した日を求める
        elapsed_month = DiffMonth(df_receipt.sales_ymd, df_customer.application_date),
    };

query.Take(10).DisplayTable()
// 考察：２つの日付の 月の差 を計算する場合、月末の扱いが 多少異なる

customer_id,sales_ymd,application_date,elapsed_month
CS006214000001,20181103,20150201,45
CS008415000097,20181118,20150322,43
CS028414000014,20170712,20150711,24
CS025415000050,20180821,20160131,30
CS003515000195,20190605,20150306,50
CS024514000042,20181205,20151010,37
CS040415000178,20190922,20150627,50
CS027514000015,20191010,20151101,47
CS025415000134,20190918,20150720,49
CS021515000126,20171010,20150508,29


---
> P-072: レシート明細データフレーム（df_receipt）の売上日（sales_ymd）に対し、顧客データフレーム（df_customer）の会員申込日（application_date）からの経過年数を計算し、顧客ID（customer_id）、売上日、会員申込日とともに表示せよ。結果は10件表示させれば良い。（なお、sales_ymdは数値、application_dateは文字列でデータを保持している点に注意）。1年未満は切り捨てること。

In [84]:
DateTime ?YYYYMMDDToDateTime(string yyyymmdd) {
    if (yyyymmdd != null && yyyymmdd.Length == 8) {
        return new DateTime(
            int.Parse(yyyymmdd.Substring(0,4)),
            int.Parse(yyyymmdd.Substring(4,2)),
            int.Parse(yyyymmdd.Substring(6)));
    } else {
        return null;
    }
}
// ２つの日付の差の月を計算
int? DiffYear(string yyyymmdd1 , string yyyymmdd2) {
    DateTime ?t1 = YYYYMMDDToDateTime(yyyymmdd1);
    DateTime ?t2 = YYYYMMDDToDateTime(yyyymmdd2);
    if (t1 != null && t2 != null) {
        if (t1.Value > t2.Value) {
            (t1,t2) = (t2,t1); // 値の入れ替え
        }
        int diff_year = t2.Value.Year - t1.Value.Year;
        // ２つの日付の月の差
        int mmdd1 = t1.Value.Month * 100 + t1.Value.Day;
        int mmdd2 = t2.Value.Month * 100 + t2.Value.Day;
        // 1年未満の場合：１減らす
        if (mmdd1> mmdd2) {
            diff_year --;
        }
        return diff_year;
    } else {
        return null;
    }
}

// レシート明細 join 顧客 
var query = from df_receipt in df_receipt
    join df_customer in df_customer on df_receipt.customer_id equals df_customer.customer_id
    select new {
        df_receipt.customer_id,
        df_receipt.sales_ymd,
        df_customer.application_date,
        // 経過した年を求める：1年未満は切り捨て
        elapsed_year = DiffYear(df_receipt.sales_ymd, df_customer.application_date),
    };

query.Take(10).DisplayTable()

customer_id,sales_ymd,application_date,elapsed_year
CS006214000001,20181103,20150201,3
CS008415000097,20181118,20150322,3
CS028414000014,20170712,20150711,2
CS025415000050,20180821,20160131,2
CS003515000195,20190605,20150306,4
CS024514000042,20181205,20151010,3
CS040415000178,20190922,20150627,4
CS027514000015,20191010,20151101,3
CS025415000134,20190918,20150720,4
CS021515000126,20171010,20150508,2


---
> P-073: レシート明細データフレーム（df_receipt）の売上日（sales_ymd）に対し、顧客データフレーム（df_customer）の会員申込日（application_date）からのエポック秒による経過時間を計算し、顧客ID（customer_id）、売上日、会員申込日とともに表示せよ。結果は10件表示させれば良い（なお、sales_ymdは数値、application_dateは文字列でデータを保持している点に注意）。なお、時間情報は保有していないため各日付は0時0分0秒を表すものとする。

In [85]:
DateTime ?YYYYMMDDToDateTime(string yyyymmdd) {
    if (yyyymmdd != null && yyyymmdd.Length == 8) {
        return new DateTime(
            int.Parse(yyyymmdd.Substring(0,4)),
            int.Parse(yyyymmdd.Substring(4,2)),
            int.Parse(yyyymmdd.Substring(6)));
    } else {
        return null;
    }
}
// ２つの日付の差の秒を計算
double? DiffSeconds(string yyyymmdd1 , string yyyymmdd2) {
    DateTime ?t1 = YYYYMMDDToDateTime(yyyymmdd1);
    DateTime ?t2 = YYYYMMDDToDateTime(yyyymmdd2);
    if (t1 != null && t2 != null) {
        if (t1.Value > t2.Value) {
            (t1,t2) = (t2,t1); // 値の入れ替え
        }
        double diffSeconds = t2.Value.Subtract(t1.Value).TotalSeconds;
        return diffSeconds;
    } else {
        return null;
    }
}

// レシート明細 join 顧客 
var query = from df_receipt in df_receipt
    join df_customer in df_customer on df_receipt.customer_id equals df_customer.customer_id
    select new {
        df_receipt.customer_id,
        df_receipt.sales_ymd,
        df_customer.application_date,
        // 経過した秒を求める
        elapsed_seconds = DiffSeconds(df_receipt.sales_ymd, df_customer.application_date),
    };

query.Take(10).DisplayTable()

customer_id,sales_ymd,application_date,elapsed_seconds
CS006214000001,20181103,20150201,118454400
CS008415000097,20181118,20150322,115516800
CS028414000014,20170712,20150711,63244800
CS025415000050,20180821,20160131,80611200
CS003515000195,20190605,20150306,134092800
CS024514000042,20181205,20151010,99532800
CS040415000178,20190922,20150627,133747200
CS027514000015,20191010,20151101,124329600
CS025415000134,20190918,20150720,131414400
CS021515000126,20171010,20150508,76550400


---
> P-074: レシート明細データフレーム（df_receipt）の売上日（sales_ymd）に対し、当該週の月曜日からの経過日数を計算し、売上日、当該週の月曜日付とともに表示せよ。結果は10件表示させれば良い（なお、sales_ymdは数値でデータを保持している点に注意）。

In [86]:
// 該当週の月曜日 と その日からの経過日数を求める
static (DateTime, int) DiffMonday(this DateTime tm) {
    // 曜日によって 月曜日の日付と 月曜日からの経過日数を求める
    var diff = tm.DayOfWeek switch {
        // 月曜日(1)の場合0
        DayOfWeek.Monday => (tm , 0),
        // 日曜日(0)の場合は 6日前の日 、 1日 経過
        DayOfWeek.Sunday => (tm.AddDays(-6) , 1),
        // 火曜(2) の場合は 月曜は 1日前 (DayOfWeek - 1) , 経過日数は dayOfWeek - 1
        // 水(3) ～土(6) も同様の計算となる
        _ => ( tm.AddDays(- ((int) tm.DayOfWeek - 1)), (int) tm.DayOfWeek - 1)
    };

    return diff;
}
DateTime ?YYYYMMDDToDateTime(string yyyymmdd) {
    if (yyyymmdd != null && yyyymmdd.Length == 8) {
        return new DateTime(
            int.Parse(yyyymmdd.Substring(0,4)),
            int.Parse(yyyymmdd.Substring(4,2)),
            int.Parse(yyyymmdd.Substring(6)));
    } else {
        return null;
    }
}

df_receipt.Select(r => new {
    r.customer_id,
    r.sales_ymd,
    diff_monday = YYYYMMDDToDateTime(r.sales_ymd)?.DiffMonday(),
}).Select(r => new {
    r.customer_id,
    r.sales_ymd,
    monday = r.diff_monday?.Item1,
    elapsed_weekday = r.diff_monday?.Item2 + " days",
}).Take(10).DisplayTable()

customer_id,sales_ymd,monday,elapsed_weekday
CS006214000001,20181103,2018-10-29,5 days
CS008415000097,20181118,2018-11-12,1 days
CS028414000014,20170712,2017-07-10,2 days
ZZ000000000000,20190205,2019-02-04,1 days
CS025415000050,20180821,2018-08-20,1 days
CS003515000195,20190605,2019-06-03,2 days
CS024514000042,20181205,2018-12-03,2 days
CS040415000178,20190922,2019-09-16,1 days
ZZ000000000000,20170504,2017-05-01,3 days
CS027514000015,20191010,2019-10-07,3 days


---
> P-075: 顧客データフレーム（df_customer）からランダムに1%のデータを抽出し、先頭から10件データを抽出せよ。

In [87]:
static IEnumerable<T> GetRandumItems<T>(this IEnumerable<T> items, double getRate) {
    // 全要素の中から ランダムにデータをピックアップして 
    // 指定のデータ件数を取得する
    int count = items.Count();
    // 最大データ取得件数：合計 * 取得データ率 : 1.0 ですべて 0.1 で 10％ 
    int get_count = (int)(getRate * count);
    Random r = new Random();
    for(int i=0;i<get_count;i++) {
        int rand_i = r.Next(count);
        T item = items.ElementAt(rand_i);
        yield return item;
    }
}

df_customer.GetRandumItems(0.01).Take(10).DisplayTable()

customer_id,customer_name,gender_cd,gender,birth_day,age,postal_cd,address,application_store_cd,application_date,status_cd
CS029403000004,薬師丸 光博,0,男性,1972-11-01,46,279-0043,千葉県浦安市富士見**********,S12029,20141222,0-00000000-0
CS008613000133,津川 あい,1,女性,1950-12-14,68,157-0068,東京都世田谷区宇奈根**********,S13008,20161008,0-00000000-0
CS004214000078,宮原 まひる,1,女性,1989-02-17,30,165-0021,東京都中野区丸山**********,S13004,20170513,0-00000000-0
CS033605000005,猪股 雄太,0,男性,1955-12-05,63,246-0031,神奈川県横浜市瀬谷区瀬谷**********,S14033,20150425,F-20100917-E
CS030312000124,浅川 薫,1,女性,1979-10-31,39,273-0033,千葉県船橋市本郷町**********,S12030,20150806,0-00000000-0
CS003501000009,村松 英嗣,0,男性,1965-03-09,54,206-0812,東京都稲城市矢野口**********,S13003,20150529,0-00000000-0
CS016415000013,梶原 花,1,女性,1969-11-06,49,184-0013,東京都小金井市前原町**********,S13016,20150606,B-20101014-E
CS009415000024,岡本 恭子,1,女性,1975-03-02,44,158-0097,東京都世田谷区用賀**********,S13009,20150814,D-20100430-E
CS006611000007,岡野 幸子,1,女性,1956-04-14,62,224-0044,神奈川県横浜市都筑区川向町**********,S14006,20150929,0-00000000-0
CS020515000223,奥村 真悠子,1,女性,1966-11-30,52,115-0056,東京都北区西が丘**********,S13020,20150129,B-20100730-D


---
> P-076: 顧客データフレーム（df_customer）から性別（gender_cd）の割合に基づきランダムに10%のデータを層化抽出データし、性別ごとに件数を集計せよ。

In [92]:
// 取得件数を指定して ランダムにデータ抽出
static IEnumerable<T> GetRandumItemsByNum<T>(this IList<T> items, int getCount) {
    // 全要素の中から ランダムにデータをピックアップして 
    // 指定のデータ件数を取得する
    int count = items.Count();
    Random r = new Random();
    // 指定の件数だけ取得する
    for(int i=0;i<getCount;i++) {
        int rand_i = r.Next(count);
        T item = items[rand_i];
        yield return item;
    }
}

// 性別 毎の 数と 率を表示
int dataNum = df_customer.Count();
var df_gender_rate = df_customer.GroupBy(r => r.gender_cd).Select(r => new {
    gender_cd = r.Key,
    gender_count = r.Count(),
    gender_rate = (double) r.Count() / dataNum,
    get_count = (int) (r.Count() * 0.1 )
}).ToDictionary(r => r.gender_cd);
display(df_gender_rate);
// 指定の比率で ランダムにデータ抽出
var df_customer_sample_gender_0 = df_customer.Where(r => r.gender_cd == "0").ToList().GetRandumItemsByNum(df_gender_rate["0"].get_count).ToList();
var df_customer_sample_gender_1 = df_customer.Where(r => r.gender_cd == "1").ToList().GetRandumItemsByNum(df_gender_rate["1"].get_count).ToList();
var df_customer_sample_gender_9 = df_customer.Where(r => r.gender_cd == "9").ToList().GetRandumItemsByNum(df_gender_rate["9"].get_count).ToList();
// ※ ToList しないと リストの内容を確認するときにランダムの値を取得しようとして処理速度

display(new {
    count_0 = df_customer_sample_gender_0.Count(),
    count_1 = df_customer_sample_gender_1.Count(),
    count_9 = df_customer_sample_gender_9.Count(),
});

// 重複なくデータ連結するのは Union を利用
// Union の場合 重複していると判断されたデータが削除される
var df_customer_sample_gender = df_customer_sample_gender_0
    .Concat(df_customer_sample_gender_1)
    .Concat(df_customer_sample_gender_9);

// Union は重複データを削除してしまう。
var df_customer_sample_gender2 = df_customer_sample_gender_0
    .Union(df_customer_sample_gender_1)
    .Union(df_customer_sample_gender_9);

display(df_customer_sample_gender2.Count());
display(df_customer_sample_gender.Count());

display(html(b("ランダムに抽出しているため 同じデータを2回取得する場合がある")));
display(new {
    gender_0_データ件数 = df_customer_sample_gender_0.Count(),
    gender_0_重複なし_データ件数 = df_customer_sample_gender_0.Distinct().Count(),
})

key value 1 { gender_cd = 1, gender_count = 17918, gender_rate = 0.8155295616949615, get_count = 1791 } gender_cd 1 gender_count 17918 gender_rate 0.8155295616949615 get_count 1791 9 { gender_cd = 9, gender_count = 1072, gender_rate = 0.04879158891265759, get_count = 107 } gender_cd 9 gender_count 1072 gender_rate 0.04879158891265759 get_count 107 0 { gender_cd = 0, gender_count = 2981, gender_rate = 0.13567884939238087, get_count = 298 } gender_cd 0 gender_count 2981 gender_rate 0.13567884939238087 get_count 298

count_0,298
count_1,1791
count_9,107


2098

2196

ランダムに抽出しているため 同じデータを2回取得する場合がある

gender_0_データ件数,298
gender_0_重複なし_データ件数,289


In [93]:
using System.Linq;
using Plotly.NET.CSharp;

Chart.BoxPlot<int,int,string>(
    Y : df_customer_sample_gender_0.Where(r => r != null).Select(r => r.age.GetValueOrDefault()).ToList(),
    BoxPoints : Plotly.NET.StyleParam.BoxPoints.All,
    Jitter : 0.3,
    PointPos : -1.8
).Display();
Chart.BoxPlot<int,int,string>(
    Y : df_customer_sample_gender_1.Where(r => r != null).Select(r => r.age.GetValueOrDefault()).ToList(),
    BoxPoints : Plotly.NET.StyleParam.BoxPoints.All,
    Jitter : 0.3,
    PointPos : -1.8
).Display();


<!DOCTYPE html>

<!DOCTYPE html>

上記のグラフが実行された結果 このような画像が表示されます。

![画像](./p-076.png)

![画像](./p-076-2.png)

---
> P-077: レシート明細データフレーム（df_receipt）の売上金額（amount）を顧客単位に合計し、合計した売上金額の外れ値を抽出せよ。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。なお、ここでは外れ値を平均から3σ以上離れたものとする。結果は10件表示させれば良い。

In [97]:
//# skleanのpreprocessing.scaleを利用するため、標本標準偏差で計算されている
//df_sales_amount = df_receipt.query('not customer_id.str.startswith("Z")', engine='python'). \
//    groupby('customer_id').agg({'amount':'sum'}).reset_index()
//df_sales_amount['amount_ss'] = preprocessing.scale(df_sales_amount['amount'])
//df_sales_amount.query('abs(amount_ss) >= 3').head(10)
var df_tmp = df_receipt.Where(r => ! r.customer_id.StartsWith("Z"))
    .GroupBy(r => r.customer_id)
    .Select(r => new {
        customer_id = r.Key,
        sum_amount = r.Sum(t => t.amount),
    }).ToList();
var dataList = df_tmp
    .Select(r => (double) r.sum_amount).ToList();

// 3σ（サン・シグマ） : 標準正規分布の 99.7％ 信頼区間 の間から 外れている値を抽出する。
var average = dataList.Median();
var stdDev = dataList.StandardDeviation(); // 標準偏差
var diffArea = new {
    average,
    stdDev, // 標準偏差
    min = average - stdDev ,
    max = average + stdDev ,
    min_3 = average - stdDev  * 3, // 3σ の 下限値
    max_3 = average + stdDev  * 3, // 3σ の 上限値
};
// 値の表示
display(diffArea);

// 散布状態の表示
Chart.BoxPlot<int,double,string>(
    Y : dataList,
    BoxPoints : Plotly.NET.StyleParam.BoxPoints.All,
    Jitter : 0.3,
    PointPos : -1.8
).Display();

df_tmp.Where(r => r.sum_amount < diffArea.min_3 || diffArea.max_3 < r.sum_amount)
    .Take(10).DisplayTable()
// 結果 

average,1478
stdDev,2720.674038361289
min,-1242.6740383612891
max,4198.674038361289
min_3,-6684.022115083868
max_3,9640.022115083868


<!DOCTYPE html>

customer_id,sum_amount
CS021515000126,9693
CS039414000052,11438
CS014415000077,14076
CS027414000084,11907
CS041411000001,9694
CS030415000197,11972
CS009415000038,14033
CS030214000008,10357
CS021515000056,12036
CS023515000195,10688


![P-077](p-077.png)

---
> P-078: レシート明細データフレーム（df_receipt）の売上金額（amount）を顧客単位に合計し、合計した売上金額の外れ値を抽出せよ。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。なお、ここでは外れ値を第一四分位と第三四分位の差であるIQRを用いて、「第一四分位数-1.5×IQR」よりも下回るもの、または「第三四分位数+1.5×IQR」を超えるものとする。結果は10件表示させれば良い。

In [98]:
var df_sales_amount = df_receipt.Where(r => !r.customer_id.StartsWith("Z"))
    .GroupBy(r => r.customer_id)
    .Select(r => new {
        customer_id = r.Key,
        sum_amount = (double) r.Sum(t => t.amount),
    });

// https://numerics.mathdotnet.com/api/MathNet.Numerics.Statistics/Statistics.htm#Percentile
double pct25 = df_sales_amount.Select(r => r.sum_amount).Percentile(25);
double pct75 = df_sales_amount.Select(r => r.sum_amount).Percentile(75);
double iqr = pct75 - pct25;
double amount_low = pct25 - (iqr * 1.5);
double amount_hight = pct75 + (iqr * 1.5);

// 結果表示
df_sales_amount
    .Where(r => r.sum_amount < amount_low || amount_hight < r.sum_amount)
    .Take(10).DisplayTable()

customer_id,sum_amount
CS021515000126,9693
CS039414000052,11438
CS026414000062,9466
CS018205000001,8739
CS010515000120,8942
CS014415000077,14076
CS027414000084,11907
CS015415000061,8856
CS041411000001,9694
CS030415000197,11972


---
> P-079: 商品データフレーム（df_product）の各項目に対し、欠損数を確認せよ。

In [101]:
var nullCount = new {
    product_cd_null = df_product.Where(r => r.product_cd == null).Count(),
    category_major_cd = df_product.Where(r => r.category_major_cd == null).Count(),
    category_medium_cd = df_product.Where(r => r.category_medium_cd == null).Count(),
    category_small_cd = df_product.Where(r => r.category_small_cd == null).Count(),
    unit_price = df_product.Where(r => r.unit_price == null).Count(),
    unit_cost = df_product.Where(r => r.unit_cost == null).Count(),
};

nullCount

product_cd_null,0
category_major_cd,0
category_medium_cd,0
category_small_cd,0
unit_price,7
unit_cost,7


---
> P-080: 商品データフレーム（df_product）のいずれかの項目に欠損が発生しているレコードを全て削除した新たなdf_product_1を作成せよ。なお、削除前後の件数を表示させ、前設問で確認した件数だけ減少していることも確認すること。

In [102]:
// いずれかの値が null の場合に true を返す
bool AnyIsNull(Product c) {
    if (c.product_cd == null) return true;
    if (c.category_major_cd == null) return true;
    if (c.category_medium_cd == null) return true;
    if (c.category_small_cd == null) return true;
    if (c.unit_cost == null) return true;
    if (c.unit_price == null) return true;
    return false;
}

// 欠損の無い物を注sh津
var df_product_1 = df_product.Where(r => ! AnyIsNull(r)).ToList();

new {
    count_削除前 = df_product.Count(),
    count_削除後 = df_product_1.Count(),
}

count_削除前,10030
count_削除後,10023


In [104]:
// いずれかの値が null の場合に true を返す
bool AnyIsNull(Product c) {
    if (c.product_cd == null) return true;
    if (c.category_major_cd == null) return true;
    if (c.category_medium_cd == null) return true;
    if (c.category_small_cd == null) return true;
    if (c.unit_cost == null) return true;
    if (c.unit_price == null) return true;
    return false;
}

// null となっているデータを確認のため表示： 7件の null は 同じレコードで発生
df_product.Where(r => AnyIsNull(r)).ToList().DisplayTable()

product_cd,category_major_cd,category_medium_cd,category_small_cd,unit_price,unit_cost
P040802007,04,0408,040802,<null>,<null>
P050103021,05,0501,050103,<null>,<null>
P050405009,05,0504,050405,<null>,<null>
P060802026,06,0608,060802,<null>,<null>
P070202092,07,0702,070202,<null>,<null>
P080504027,08,0805,080504,<null>,<null>
P090204185,09,0902,090204,<null>,<null>


---
> P-081: 単価（unit_price）と原価（unit_cost）の欠損値について、それぞれの平均値で補完した新たなdf_product_2を作成せよ。なお、平均値について1円未満は四捨五入とし、0.5については偶数寄せでかまわない。補完実施後、各項目について欠損が生じていないことも確認すること。

In [105]:
// いずれかの値が null の場合に true を返す
bool AnyIsNull(Product c) {
    if (c.product_cd == null) return true;
    if (c.category_major_cd == null) return true;
    if (c.category_medium_cd == null) return true;
    if (c.category_small_cd == null) return true;
    if (c.unit_cost == null) return true;
    if (c.unit_price == null) return true;
    return false;
}

// https://docs.microsoft.com/ja-jp/dotnet/api/system.math.round?view=net-5.0 
// Math.Round : 偶数丸め
var fix_unit_price = (int) Math.Round(df_product.Where(r => r.unit_price != null).Average(r => r.unit_price.Value));
var fix_unit_cost = (int) Math.Round(df_product.Where(r => r.unit_cost != null).Average(r => r.unit_cost.Value));

var df_product_2 = df_product.Select(r => new Product() {
    product_cd = r.product_cd,
    category_major_cd = r.category_major_cd,
    category_medium_cd = r.category_medium_cd,
    category_small_cd = r.category_small_cd,
    unit_price = r.unit_price ?? fix_unit_price,
    unit_cost = r.unit_cost ?? fix_unit_cost,
});

// null が存在しない事の確認
df_product_2.Where(r => AnyIsNull(r)).Count()


0

---
> P-082: 単価（unit_price）と原価（unit_cost）の欠損値について、それぞれの中央値で補完した新たなdf_product_3を作成せよ。なお、中央値について1円未満は四捨五入とし、0.5については偶数寄せでかまわない。補完実施後、各項目について欠損が生じていないことも確認すること。

In [106]:
// 中央値 : Median 
//https://numerics.mathdotnet.com/api/MathNet.Numerics.Statistics/Statistics.htm#Median
using MathNet.Numerics.Statistics;

// https://docs.microsoft.com/ja-jp/dotnet/api/system.math.round?view=net-5.0 
// Math.Round : 偶数丸め

var fix_unit_price = (int) Math.Round(df_product.Where(r => r.unit_price != null).Select(r => (double) r.unit_price).Median());
var fix_unit_cost = (int) Math.Round(df_product.Where(r => r.unit_cost != null).Select(r => (double) r.unit_cost).Median());

var df_product_2 = df_product.Select(r => new Product() { // AnyIsNull で判定するため 厳密な型に合わせる
    product_cd = r.product_cd,
    category_major_cd = r.category_major_cd,
    category_medium_cd = r.category_medium_cd,
    category_small_cd = r.category_small_cd,
    unit_price = r.unit_price ?? fix_unit_price,
    unit_cost = r.unit_cost ?? fix_unit_cost,
});

// いずれかの値が null の場合に true を返す
bool AnyIsNull(Product c) {
    if (c.product_cd == null) return true;
    if (c.category_major_cd == null) return true;
    if (c.category_medium_cd == null) return true;
    if (c.category_small_cd == null) return true;
    if (c.unit_cost == null) return true;
    if (c.unit_price == null) return true;
    return false;
}

// null が存在しない事の確認
df_product_2.Where(r => AnyIsNull(r)).Count()

0

---
> P-083: 単価（unit_price）と原価（unit_cost）の欠損値について、各商品の小区分（category_small_cd）ごとに算出した中央値で補完した新たなdf_product_4を作成せよ。なお、中央値について1円未満は四捨五入とし、0.5については偶数寄せでかまわない。補完実施後、各項目について欠損が生じていないことも確認すること。

In [108]:
using MathNet.Numerics.Statistics;

var median_tmp = df_product.GroupBy(r => r.category_small_cd)
    .Select(r => new {
        category_small_cd = r.Key,
        median_unit_price = r.Where(t => t.unit_price != null).Select(t => (double) t.unit_price).Median(),
        median_unit_cost = r.Where(t => t.unit_cost != null).Select(t => (double) t.unit_cost).Median(),
    })
    .ToDictionary(r => r.category_small_cd);

display("category_small_cd 毎の中央値");
median_tmp.Take(20).DisplayTable();

var df_product4 = df_product.Select(r => new Product() {
    product_cd = r.product_cd,
    category_major_cd = r.category_major_cd,
    category_medium_cd = r.category_medium_cd,
    category_small_cd = r.category_small_cd,
    unit_price = r.unit_price ?? (int) Math.Round(median_tmp[r.category_small_cd].median_unit_price),
    unit_cost = r.unit_cost ?? (int) Math.Round(median_tmp[r.category_small_cd].median_unit_cost),
});

display("df_product4 の 表示");
df_product4.Take(20).DisplayTable();

// いずれかの値が null の場合に true を返す
bool AnyIsNull(Product c) {
    if (c.product_cd == null) return true;
    if (c.category_major_cd == null) return true;
    if (c.category_medium_cd == null) return true;
    if (c.category_small_cd == null) return true;
    if (c.unit_cost == null) return true;
    if (c.unit_price == null) return true;
    return false;
}

display($"いずれか null の データ件数 { df_product4.Where(r => AnyIsNull(r)).Count() }");

category_small_cd 毎の中央値

Key Value 040101 { category_small_cd = 040101, median_unit_price = 283, median_unit_cost = 212.5 } category_small_cd 040101 median_unit_price 283 median_unit_cost 212.5 040102 { category_small_cd = 040102, median_unit_price = 378, median_unit_cost = 284 } category_small_cd 040102 median_unit_price 378 median_unit_cost 284 040201 { category_small_cd = 040201, median_unit_price = 223, median_unit_cost = 167.5 } category_small_cd 040201 median_unit_price 223 median_unit_cost 167.5 040202 { category_small_cd = 040202, median_unit_price = 178, median_unit_cost = 134 } category_small_cd 040202 median_unit_price 178 median_unit_cost 134 040203 { category_small_cd = 040203, median_unit_price = 308, median_unit_cost = 231 } category_small_cd 040203 median_unit_price 308 median_unit_cost 231 040204 { category_small_cd = 040204, median_unit_price = 198, median_unit_cost = 149 } category_small_cd 040204 median_unit_price 198 median_unit_cost 149 040301 { category_small_cd = 040301, median_unit_price = 278, median_unit_cost = 209 } category_small_cd 040301 median_unit_price 278 median_unit_cost 209 040401 { category_small_cd = 040401, median_unit_price = 288, median_unit_cost = 216.5 } category_small_cd 040401 median_unit_price 288 median_unit_cost 216.5 040402 { category_small_cd = 040402, median_unit_price = 228, median_unit_cost = 171 } category_small_cd 040402 median_unit_price 228 median_unit_cost 171 040403 { category_small_cd = 040403, median_unit_price = 248, median_unit_cost = 186 } category_small_cd 040403 median_unit_price 248 median_unit_cost 186 040404 { category_small_cd = 040404, median_unit_price = 278, median_unit_cost = 209 } category_small_cd 040404 median_unit_price 278 median_unit_cost 209 040501 { category_small_cd = 040501, median_unit_price = 288, median_unit_cost = 216.5 } category_small_cd 040501 median_unit_price 288 median_unit_cost 216.5 040502 { category_small_cd = 040502, median_unit_price = 298, median_unit_cost = 224 } category_small_cd 040502 median_unit_price 298 median_unit_cost 224 040503 { category_small_cd = 040503, median_unit_price = 298, median_unit_cost = 224 } category_small_cd 040503 median_unit_price 298 median_unit_cost 224 040504 { category_small_cd = 040504, median_unit_price = 323, median_unit_cost = 242.5 } category_small_cd 040504 median_unit_price 323 median_unit_cost 242.5 040601 { category_small_cd = 040601, median_unit_price = 278, median_unit_cost = 209 } category_small_cd 040601 median_unit_price 278 median_unit_cost 209 040602 { category_small_cd = 040602, median_unit_price = 422.5, median_unit_cost = 317 } category_small_cd 040602 median_unit_price 422.5 median_unit_cost 317 040603 { category_small_cd = 040603, median_unit_price = 1980, median_unit_cost = 1485 } category_small_cd 040603 median_unit_price 1980 median_unit_cost 1485 040701 { category_small_cd = 040701, median_unit_price = 288, median_unit_cost = 216 } category_small_cd 040701 median_unit_price 288 median_unit_cost 216 040801 { category_small_cd = 040801, median_unit_price = 148, median_unit_cost = 111 } category_small_cd 040801 median_unit_price 148 median_unit_cost 111

df_product4 の 表示

product_cd,category_major_cd,category_medium_cd,category_small_cd,unit_price,unit_cost
P040101001,04,0401,040101,198,149
P040101002,04,0401,040101,218,164
P040101003,04,0401,040101,230,173
P040101004,04,0401,040101,248,186
P040101005,04,0401,040101,268,201
P040101006,04,0401,040101,298,224
P040101007,04,0401,040101,338,254
P040101008,04,0401,040101,420,315
P040101009,04,0401,040101,498,374
P040101010,04,0401,040101,580,435


いずれか null の データ件数 0

---
> P-084: 顧客データフレーム（df_customer）の全顧客に対し、全期間の売上金額に占める2019年売上金額の割合を計算せよ。ただし、販売実績のない場合は0として扱うこと。そして計算した割合が0超のものを抽出せよ。 結果は10件表示させれば良い。また、作成したデータにNAやNANが存在しないことを確認せよ。

In [109]:
// 顧客データフレーム（df_customer）の全顧客に対し、全期間の売上金額に占める2019年売上金額の割合を計算せよ。
// ただし、販売実績のない場合は0として扱うこと。そして計算した割合が0超のものを抽出せよ。 
// 結果は10件表示させれば良い。また、作成したデータにNAやNANが存在しないことを確認せよ。

// 対象となる顧客は 2019年度に売上のある顧客
var df_tmp_1 = df_receipt.Where(r => r.sales_ymd.Substring(0,4) == "2019")
    .GroupBy(r => r.customer_id)
    .Select(r => new {
        customer_id = r.Key,
        sum_amount_2019 = r.Sum(t => t.amount),
    });
// 全期間の顧客毎の売り上げ
var df_sum_amount_dic = df_receipt
.GroupBy(r => r.customer_id)
.Select(r => new {
    customer_id = r.Key,
    sum_amount = r.Sum(t => t.amount ?? 0),
}).ToDictionary(r => r.customer_id);

// 割合を計算する
var df_ans = df_tmp_1
    .Where(r => r.sum_amount_2019 > 0)  // 2019年に0以上の売り上げがある人
    .Select(r => new {
    r.customer_id,
    r.sum_amount_2019,
    rate_amount = (double) r.sum_amount_2019 / df_sum_amount_dic[r.customer_id].sum_amount,
}).OrderBy(r => r.customer_id).ToList();

// NA や NAN が無い事を確認する
var null_count = df_ans.Where(r => r.sum_amount_2019 == null).Count();

df_ans.Take(10).DisplayTable();
display($"Null の件数 { null_count }")

customer_id,sum_amount_2019,rate_amount
CS001113000004,1298,1
CS001114000005,188,0.3003194888178914
CS001115000010,578,0.18988173455978974
CS001205000004,702,0.35311871227364183
CS001205000006,486,0.14563979622415343
CS001211000025,456,1
CS001212000070,456,1
CS001214000009,664,0.14172892209178228
CS001214000017,2962,0.7168441432720233
CS001214000048,1889,0.7957034540859309


Null の件数 0

---
> P-085: 顧客データフレーム（df_customer）の全顧客に対し、郵便番号（postal_cd）を用いて経度緯度変換用データフレーム（df_geocode）を紐付け、新たなdf_customer_1を作成せよ。ただし、複数紐づく場合は経度（longitude）、緯度（latitude）それぞれ平均を算出すること。

In [111]:
// 郵便番号と それに紐づく 経度（longitude）、緯度（latitude） の平均の値 : 郵便番号をキーに検索できる Dictionary 型データ
var df_geocode_avg_dic = df_geocode.GroupBy(r => r.postal_cd)
    .Select(r => new {
        postal_cd = r.Key,
        m_longitude = r.Average(t => t.longitude),
        m_latitude = r.Average(t => t.latitude),
    })
    .ToDictionary(r => r.postal_cd);

var df_customer_1 = df_customer.Select(r => new {
    r.customer_id,
    r.customer_name,
    r.gender_cd,
    r.gender,
    r.birth_day,
    r.age,
    r.postal_cd,
    r.address,
    r.application_store_cd,
    r.application_date,
    r.status_cd,
    m_longitude = df_geocode_avg_dic[r.postal_cd].m_longitude,
    m_latitude = df_geocode_avg_dic[r.postal_cd].m_latitude,
}).ToList();

df_customer_1.DisplayTable();

// customer_id	customer_name	gender_cd	gender	birth_day	age	postal_cd	address	application_store_cd	application_date	status_cd	age_group	m_longitude	m_latitude

customer_id,customer_name,gender_cd,gender,birth_day,age,postal_cd,address,application_store_cd,application_date,status_cd,m_longitude,m_latitude
CS021313000114,大野 あや子,1,女性,1981-04-29,37,259-1113,神奈川県伊勢原市粟窪**********,S14021,20150905,0-00000000-0,139.31779,35.41358
CS037613000071,六角 雅彦,9,不明,1952-04-01,66,136-0076,東京都江東区南砂**********,S13037,20150414,0-00000000-0,139.83502,35.67193
CS031415000172,宇多田 貴美子,1,女性,1976-10-04,42,151-0053,東京都渋谷区代々木**********,S13031,20150529,D-20100325-C,139.68965,35.67374
CS028811000001,堀井 かおり,1,女性,1933-03-27,86,245-0016,神奈川県横浜市泉区和泉町**********,S14028,20160115,0-00000000-0,139.4836,35.39125
CS001215000145,田崎 美紀,1,女性,1995-03-29,24,144-0055,東京都大田区仲六郷**********,S13001,20170605,6-20090929-2,139.70775,35.54084
CS020401000016,宮下 達士,0,男性,1974-09-15,44,174-0065,東京都板橋区若木**********,S13020,20150225,0-00000000-0,139.67245,35.77073
CS015414000103,奥野 陽子,1,女性,1977-08-09,41,136-0073,東京都江東区北砂**********,S13015,20150722,B-20100609-B,139.83601,35.67818
CS029403000008,釈 人志,0,男性,1973-08-17,45,279-0003,千葉県浦安市海楽**********,S12029,20150515,0-00000000-0,139.90469,35.65422
CS015804000004,松谷 米蔵,0,男性,1931-05-02,87,136-0073,東京都江東区北砂**********,S13015,20150607,0-00000000-0,139.83601,35.67818
CS033513000180,安斎 遥,1,女性,1962-07-11,56,241-0823,神奈川県横浜市旭区善部町**********,S14033,20150728,6-20080506-5,139.51463,35.45013


---
> P-086: 前設問で作成した緯度経度つき顧客データフレーム（df_customer_1）に対し、申込み店舗コード（application_store_cd）をキーに店舗データフレーム（df_store）と結合せよ。そして申込み店舗の緯度（latitude）・経度情報（longitude)と顧客の緯度・経度を用いて距離（km）を求め、顧客ID（customer_id）、顧客住所（address）、店舗住所（address）とともに表示せよ。計算式は簡易式で良いものとするが、その他精度の高い方式を利用したライブラリを利用してもかまわない。結果は10件表示すれば良い。

```
緯度（ラジアン）：\phi \\
経度（ラジアン）：\lambda \\
距離L = 6371 * arccos(sin \phi_1 * sin \phi_2
+ cos \phi_1 * cos \phi_2 * cos(\lambda_1 − \lambda_2))
```

In [112]:
var df_store_dic = df_store.ToDictionary(r => r.store_cd);

//def calc_distance(x1, y1, x2, y2):
//    x1_r = np.radians(x1)
//    x2_r = np.radians(x2)
//    y1_r = np.radians(y1)
//    y2_r = np.radians(y2)
//    return 6371 * np.arccos(np.sin(y1_r) * np.sin(y2_r) 
//                            + np.cos(y1_r) * np.cos(y2_r) 
//                            * np.cos(x1_r - x2_r))
//    return distance
double DegreeToRadian(double deg) {
    return Math.PI * deg / 180.0;
}
double CalcDistance(double x1, double y1, double x2, double y2) {
    double x1_r = DegreeToRadian(x1);
    double x2_r = DegreeToRadian(x2);
    double y1_r = DegreeToRadian(y1);
    double y2_r = DegreeToRadian(y2);

    return 6371 * Math.Acos(Math.Sin(y1_r) * Math.Sin(y2_r) 
                    + Math.Cos(y1_r) * Math.Cos(y2_r) 
                        * Math.Cos(x1_r - x2_r));
}

df_customer_1.Select(r => new {
    r.customer_id,
    r.address,
    store_address = df_store_dic[r.application_store_cd].address,
    //r.m_latitude,
    //r.m_longitude,
    //application_store_latitude = df_store_dic[r.application_store_cd].latitude,
    //application_store_longitude = df_store_dic[r.application_store_cd].longitude,
    distance = CalcDistance((double) r.m_longitude, (double) r.m_latitude, (double) df_store_dic[r.application_store_cd].longitude, (double) df_store_dic[r.application_store_cd].latitude),
}).Take(10).DisplayTable()

customer_id,address,store_address,distance
CS021313000114,神奈川県伊勢原市粟窪**********,神奈川県伊勢原市伊勢原四丁目,1.3944087027154715
CS037613000071,東京都江東区南砂**********,東京都江東区南砂一丁目,1.4511822099658445
CS031415000172,東京都渋谷区代々木**********,東京都渋谷区初台二丁目,0.4117334789298223
CS028811000001,神奈川県横浜市泉区和泉町**********,神奈川県横浜市瀬谷区二ツ橋町,8.065196026704987
CS001215000145,東京都大田区仲六郷**********,東京都大田区仲六郷二丁目,1.2684209720729687
CS020401000016,東京都板橋区若木**********,東京都北区十条仲原三丁目,4.18590461529678
CS015414000103,東京都江東区北砂**********,東京都江東区南砂二丁目,1.4496734114236336
CS029403000008,千葉県浦安市海楽**********,千葉県浦安市東野一丁目,0.804858125326335
CS015804000004,東京都江東区北砂**********,東京都江東区南砂二丁目,1.4496734114236336
CS033513000180,神奈川県横浜市旭区善部町**********,神奈川県横浜市瀬谷区阿久和西一丁目,1.9569470032302752


---
> P-087:  顧客データフレーム（df_customer）では、異なる店舗での申込みなどにより同一顧客が複数登録されている。名前（customer_name）と郵便番号（postal_cd）が同じ顧客は同一顧客とみなし、1顧客1レコードとなるように名寄せした名寄顧客データフレーム（df_customer_u）を作成せよ。ただし、同一顧客に対しては売上金額合計が最も高いものを残すものとし、売上金額合計が同一もしくは売上実績の無い顧客については顧客ID（customer_id）の番号が小さいものを残すこととする。

In [113]:
//df_tmp = df_receipt.groupby('customer_id').agg({'amount':sum}).reset_index()
//df_customer_u = pd.merge(df_customer, df_tmp, how='left', on='customer_id').sort_values(['amount', 'customer_id']
//                                                                                        , ascending=[False, True])
//df_customer_u.drop_duplicates(subset=['customer_name', 'postal_cd'], keep='first', inplace=True)
//
//print('減少数: ', len(df_customer) - len(df_customer_u))
// 顧客ID 毎の売上金額の計算
var df_amount_dic = df_receipt.GroupBy(r => r.customer_id).Select(r => new {
    customer_id = r.Key,
    sum_amount = r.Sum(t => t.amount),
}).ToDictionary(r => r.customer_id);

var df_customer_u = df_customer
    .Select(r => new {
        r.customer_id,
        nayose_key = r.customer_name + "\t" + r.postal_cd,
        sum_amount = df_amount_dic.GetValueOrDefault(r.customer_id)?.sum_amount ?? 0,
    })
    .GroupBy(r => r.nayose_key)
    .Select(r => new {
        nayose_key = r.Key,
        customer_id = r.OrderByDescending(t => t.sum_amount).FirstOrDefault()?.customer_id,
    })
    .ToList();

// 現象数をカウント
df_customer.Count() - df_customer_u.Count()

30

---
> P-088: 前設問で作成したデータを元に、顧客データフレームに統合名寄IDを付与したデータフレーム（df_customer_n）を作成せよ。ただし、統合名寄IDは以下の仕様で付与するものとする。
>
> - 重複していない顧客：顧客ID（customer_id）を設定
> - 重複している顧客：前設問で抽出したレコードの顧客IDを設定

In [116]:
//df_customer_n = pd.merge(df_customer, df_customer_u[['customer_name', 'postal_cd', 'customer_id']],
//                        how='inner', on =['customer_name', 'postal_cd'])
//df_customer_n.rename(columns={'customer_id_x':'customer_id', 'customer_id_y':'integration_id'}, inplace=True)
//
//print('ID数の差', len(df_customer_n['customer_id'].unique()) - len(df_customer_n['integration_id'].unique()))
var nayose_customer_id_dic = df_customer_u.ToDictionary(r => r.nayose_key);

var df_customer_n = df_customer.Select(r => new {
    customer_id = nayose_customer_id_dic[r.customer_name + "\t" + r.postal_cd].customer_id,
    r.customer_name,
    r.gender_cd, r.gender,
    r.postal_cd,r.address,
    r.age,
    r.application_date,
    r.application_store_cd,
    r.birth_day,
    r.status_cd,
});

df_customer_n.Take(10).DisplayTable()

customer_id,customer_name,gender_cd,gender,postal_cd,address,age,application_date,application_store_cd,birth_day,status_cd
CS021313000114,大野 あや子,1,女性,259-1113,神奈川県伊勢原市粟窪**********,37,20150905,S14021,1981-04-29,0-00000000-0
CS037613000071,六角 雅彦,9,不明,136-0076,東京都江東区南砂**********,66,20150414,S13037,1952-04-01,0-00000000-0
CS031415000172,宇多田 貴美子,1,女性,151-0053,東京都渋谷区代々木**********,42,20150529,S13031,1976-10-04,D-20100325-C
CS028811000001,堀井 かおり,1,女性,245-0016,神奈川県横浜市泉区和泉町**********,86,20160115,S14028,1933-03-27,0-00000000-0
CS001215000145,田崎 美紀,1,女性,144-0055,東京都大田区仲六郷**********,24,20170605,S13001,1995-03-29,6-20090929-2
CS020401000016,宮下 達士,0,男性,174-0065,東京都板橋区若木**********,44,20150225,S13020,1974-09-15,0-00000000-0
CS015414000103,奥野 陽子,1,女性,136-0073,東京都江東区北砂**********,41,20150722,S13015,1977-08-09,B-20100609-B
CS029403000008,釈 人志,0,男性,279-0003,千葉県浦安市海楽**********,45,20150515,S12029,1973-08-17,0-00000000-0
CS015804000004,松谷 米蔵,0,男性,136-0073,東京都江東区北砂**********,87,20150607,S13015,1931-05-02,0-00000000-0
CS033513000180,安斎 遥,1,女性,241-0823,神奈川県横浜市旭区善部町**********,56,20150728,S14033,1962-07-11,6-20080506-5


---
> P-089: 売上実績のある顧客に対し、予測モデル構築のため学習用データとテスト用データに分割したい。それぞれ8:2の割合でランダムにデータを分割せよ。

In [117]:
// df_tmp = pd.merge(df_customer, df_receipt['customer_id'], how='inner', on='customer_id')
// df_train, df_test = train_test_split(df_tmp, test_size=0.2, random_state=71)
// print('学習データ割合: ', len(df_train) / len(df_tmp))
// print('テストデータ割合: ', len(df_test) / len(df_tmp))


// 指定の件数になるまでデータを抽出する
static HashSet<string> GetListRandum(List<string> items, int getSize) {
    // 全要素の中から ランダムにデータをピックアップして 
    // 指定のデータ件数になるまで繰り返す
    Random r = new Random();
    int count = items.Count();
    if (count < getSize) {
        getSize = count;
    }
    HashSet<string> ansSet = new HashSet<string>();
    while(ansSet.Count < getSize) {
        var ix = r.Next(count);
        string id = items[ix];
        ansSet.Add(id);
    }

    return ansSet;
}

// 売上実績のある顧客の抽出
var query_cust = from df_receipt in df_receipt 
    join df_customer in df_customer on df_receipt.customer_id equals df_customer.customer_id
    select df_customer.customer_id;

// 対象となる顧客
var targetCustmerList = query_cust.Distinct().ToList();
int pct20 = (int) (targetCustmerList.Count() * 0.2);
// 20％ に含まれる人
var df_cusomer_20 = GetListRandum(targetCustmerList, pct20);
// 残りの 80% 
var df_cusomer_80 = targetCustmerList.Except(df_cusomer_20).ToHashSet();

//print(nrow(df_customer_train))
//print(nrow(df_customer_test))
// 80% の人が含まれている
var df_customer_train = df_customer.Where(r => df_cusomer_80.Contains(r.customer_id)).ToList();
// 20% の人が含まれている
var df_customer_test = df_customer.Where(r => df_cusomer_20.Contains(r.customer_id)).ToList();

new {
    train_size = df_customer_train.Count(),
    test_size = df_customer_test.Count(),
}

train_size,6645
test_size,1661


---
> P-090: レシート明細データフレーム（df_receipt）は2017年1月1日〜2019年10月31日までのデータを有している。売上金額（amount）を月次で集計し、学習用に12ヶ月、テスト用に6ヶ月のモデル構築用データを3セット作成せよ。

In [118]:
//df_tmp = df_receipt[['sales_ymd', 'amount']].copy()
//df_tmp['sales_ym'] = df_tmp['sales_ymd'].astype('str').str[0:6]
//df_tmp = df_tmp.groupby('sales_ym').agg({'amount':'sum'}).reset_index()

//# 関数化することで長期間データに対する多数のデータセットもループなどで処理できるようにする
//def split_data(df, train_size, test_size, slide_window, start_point):
//    train_start = start_point * slide_window
//    test_start = train_start + train_size
//    return df[train_start : test_start], df[test_start : test_start + test_size]

//df_train_1, df_test_1 = split_data(df_tmp, train_size=12, test_size=6, slide_window=6, start_point=0)
//df_train_2, df_test_2 = split_data(df_tmp, train_size=12, test_size=6, slide_window=6, start_point=1)
//df_train_3, df_test_3 = split_data(df_tmp, train_size=12, test_size=6, slide_window=6, start_point=2)

new {
    max = df_receipt.Max(r => r.sales_ymd),
    min = df_receipt.Min(r => r.sales_ymd),
}

max,20191031
min,20170101


---
> P-091: 顧客データフレーム（df_customer）の各顧客に対し、売上実績のある顧客数と売上実績のない顧客数が1:1となるようにアンダーサンプリングで抽出せよ。

In [119]:
// 顧客単位の売り上げを計算
var cus_amount = df_receipt.GroupBy(r => r.customer_id).Select(r => new {
    customer_id = r.Key,
    sum_amount = r.Sum(t => t.amount),
}).ToDictionary(r => r.customer_id);

// 売上実績のある顧客
var amount_customer = df_receipt.Where(r => r.amount > 0).Select(r => r.customer_id).Distinct().ToHashSet();
// 売上実績のない顧客
var no_amount_customer = df_customer.Where(r => ! amount_customer.Contains(r.customer_id)).Select(r => r.customer_id).ToHashSet();

display(amount_customer.Count());

int get_n = Math.Min(amount_customer.Count(), no_amount_customer.Count());
// 条件を指定してランダムにデータ件数抽出

// 指定の件数になるまでデータを抽出する
static HashSet<string> GetListRandumOrAll(HashSet<string> items, int getSize) {
    // 求める件数より多い場合はすべてを返す
    if (getSize >= items.Count) {
        return items;
    }
    // 全要素の中から ランダムにデータをピックアップして 
    // 指定のデータ件数になるまで繰り返す
    Random r = new Random();
    int count = items.Count();
    if (count < getSize) {
        getSize = count;
    }
    HashSet<string> ansSet = new HashSet<string>();
    var list = items.ToList();
    while(ansSet.Count < getSize) {
        var ix = r.Next(count);
        string id = list[ix];
        ansSet.Add(id);
    }

    return ansSet;
}

var sample_amount_cust_id = GetListRandumOrAll(amount_customer, get_n);
var sample_no_amount_cust_id = GetListRandumOrAll(no_amount_customer, get_n);

display(new {
    売上実績のある顧客_件数 = sample_amount_cust_id.Count(),
    売上実績のない顧客_件数 = sample_no_amount_cust_id.Count()
});

8307

売上実績のある顧客_件数,8307
売上実績のない顧客_件数,8307


---
> P-092: 顧客データフレーム（df_customer）では、性別に関する情報が非正規化の状態で保持されている。これを第三正規化せよ。

In [122]:
public record Gender {
    public string gender_cd {get; init;}
    public string gender {get;init;}
}

public class CustomerS
{
    public string customer_id { get; init; }
    public string customer_name { get; init; }
    public DateTime? birth_day { get; init; }
    public int? age { get; init; }
    public string postal_cd { get; init; }
    public string address { get; init; }
    public string application_store_cd { get; init; }
    public string application_date { get; init; }
    public string status_cd { get; init; }

    public Gender gender {get;init;}
}

var df_gender = df_customer.GroupBy(r => r.gender_cd)
    .Select(r => new Gender() {
        gender_cd = r.Key,
        gender = r.Select(t => t.gender).FirstOrDefault(),
    })
    .ToDictionary(r => r.gender_cd);

var df_customer_s = df_customer.Select(r => new CustomerS() {
    customer_id = r.customer_id,
    customer_name = r.customer_name,
    birth_day = r.birth_day,
    age = r.age,
    address = r.address,
    postal_cd = r.postal_cd,
    application_store_cd = r.application_store_cd,
    application_date = r.application_date,
    status_cd = r.status_cd,
    gender = df_gender.GetValueOrDefault(r.gender_cd),
}).ToList();

df_customer_s.DisplayTable()

customer_id customer_name birth_day age postal_cd address application_store_cd application_date status_cd gender CS021313000114 大野 あや子 1981-04-29 37 259-1113 神奈川県伊勢原市粟窪********** S14021 20150905 0-00000000-0 Gender { gender_cd = 1, gender = 女性 } gender_cd 1 gender 女性 CS037613000071 六角 雅彦 1952-04-01 66 136-0076 東京都江東区南砂********** S13037 20150414 0-00000000-0 Gender { gender_cd = 9, gender = 不明 } gender_cd 9 gender 不明 CS031415000172 宇多田 貴美子 1976-10-04 42 151-0053 東京都渋谷区代々木********** S13031 20150529 D-20100325-C Gender { gender_cd = 1, gender = 女性 } gender_cd 1 gender 女性 CS028811000001 堀井 かおり 1933-03-27 86 245-0016 神奈川県横浜市泉区和泉町********** S14028 20160115 0-00000000-0 Gender { gender_cd = 1, gender = 女性 } gender_cd 1 gender 女性 CS001215000145 田崎 美紀 1995-03-29 24 144-0055 東京都大田区仲六郷********** S13001 20170605 6-20090929-2 Gender { gender_cd = 1, gender = 女性 } gender_cd 1 gender 女性 CS020401000016 宮下 達士 1974-09-15 44 174-0065 東京都板橋区若木********** S13020 20150225 0-00000000-0 Gender { gender_cd = 0, gender = 男性 } gender_cd 0 gender 男性 CS015414000103 奥野 陽子 1977-08-09 41 136-0073 東京都江東区北砂********** S13015 20150722 B-20100609-B Gender { gender_cd = 1, gender = 女性 } gender_cd 1 gender 女性 CS029403000008 釈 人志 1973-08-17 45 279-0003 千葉県浦安市海楽********** S12029 20150515 0-00000000-0 Gender { gender_cd = 0, gender = 男性 } gender_cd 0 gender 男性 CS015804000004 松谷 米蔵 1931-05-02 87 136-0073 東京都江東区北砂********** S13015 20150607 0-00000000-0 Gender { gender_cd = 0, gender = 男性 } gender_cd 0 gender 男性 CS033513000180 安斎 遥 1962-07-11 56 241-0823 神奈川県横浜市旭区善部町********** S14033 20150728 6-20080506-5 Gender { gender_cd = 1, gender = 女性 } gender_cd 1 gender 女性 CS007403000016 依田 満 1975-08-18 43 276-0022 千葉県八千代市上高野********** S12007 20150914 0-00000000-0 Gender { gender_cd = 0, gender = 男性 } gender_cd 0 gender 男性 CS035614000014 板倉 菜々美 1954-07-16 64 154-0015 東京都世田谷区桜新町********** S13035 20150804 0-00000000-0 Gender { gender_cd = 1, gender = 女性 } gender_cd 1 gender 女性 CS011215000048 芦田 沙耶 1992-02-01 27 223-0062 神奈川県横浜市港北区日吉本町********** S14011 20150228 C-20100421-9 Gender { gender_cd = 1, gender = 女性 } gender_cd 1 gender 女性 CS009413000079 市川 コウ 1975-12-28 43 158-0093 東京都世田谷区上野毛********** S13009 20151209 0-00000000-0 Gender { gender_cd = 1, gender = 女性 } gender_cd 1 gender 女性 CS040412000191 川井 郁恵 1977-01-05 42 226-0021 神奈川県横浜市緑区北八朔町********** S14040 20151101 1-20091025-4 Gender { gender_cd = 1, gender = 女性 } gender_cd 1 gender 女性 CS029415000023 梅田 里穂 1976-01-17 43 279-0043 千葉県浦安市富士見********** S12029 20150610 D-20100918-E Gender { gender_cd = 1, gender = 女性 } gender_cd 1 gender 女性 CS009315000023 皆川 文世 1980-04-15 38 154-0012 東京都世田谷区駒沢********** S13009 20150319 5-20080322-1 Gender { gender_cd = 1, gender = 女性 } gender_cd 1 gender 女性 CS040702000012 根本 六郎 1939-07-02 79 226-0018 神奈川県横浜市緑区長津田みなみ台********** S14040 20150112 0-00000000-0 Gender { gender_cd = 0, gender = 男性 } gender_cd 0 gender 男性 CS046615000013 河野 花 1953-04-06 65 224-0026 神奈川県横浜市都筑区南山田町********** S14046 20181207 0-00000000-0 Gender { gender_cd = 1, gender = 女性 } gender_cd 1 gender 女性 CS025412000147 堀口 陽子 1974-10-22 44 242-0015 神奈川県大和市下和田********** S14025 20150417 0-00000000-0 Gender { gender_cd = 1, gender = 女性 } gender_cd 1 gender 女性 (21951 more)

---
> P-093: 商品データフレーム（df_product）では各カテゴリのコード値だけを保有し、カテゴリ名は保有していない。カテゴリデータフレーム（df_category）と組み合わせて非正規化し、カテゴリ名を保有した新たな商品データフレームを作成せよ。

In [123]:
/// Prodoct と Category の 項目を 持つクラス
public class ProductFull
{
    public string product_cd { get; set; }
    public string category_major_cd { get; set; }
    public string category_major_name { get; set; }
    public string category_medium_cd { get; set; }
    public string category_medium_name { get; set; }
    public string category_small_cd { get; set; }
    public string category_small_name { get; set; }
    public int? unit_price { get; set; }
    public int? unit_cost { get; set; }
}

// category_small_cd で Join した結合データを作成
var query = from df_category in df_category 
            join df_product in df_product 
            on df_category.category_small_cd equals df_product.category_small_cd
            select new ProductFull() {
                product_cd = df_product.product_cd,
                category_major_cd = df_product.category_major_cd,
                category_medium_cd = df_product.category_medium_cd,
                category_small_cd = df_product.category_small_cd,
                unit_price = df_product.unit_price,
                unit_cost = df_product.unit_cost,
                category_major_name = df_category.category_major_name,
                category_medium_name = df_category.category_medium_name,
                category_small_name = df_category.category_small_name,
            };

var df_product_full = query.ToList();

df_product_full.DisplayTable();

product_cd,category_major_cd,category_major_name,category_medium_cd,category_medium_name,category_small_cd,category_small_name,unit_price,unit_cost
P040101001,04,惣菜,0401,御飯類,040101,弁当類,198,149
P040101002,04,惣菜,0401,御飯類,040101,弁当類,218,164
P040101003,04,惣菜,0401,御飯類,040101,弁当類,230,173
P040101004,04,惣菜,0401,御飯類,040101,弁当類,248,186
P040101005,04,惣菜,0401,御飯類,040101,弁当類,268,201
P040101006,04,惣菜,0401,御飯類,040101,弁当類,298,224
P040101007,04,惣菜,0401,御飯類,040101,弁当類,338,254
P040101008,04,惣菜,0401,御飯類,040101,弁当類,420,315
P040101009,04,惣菜,0401,御飯類,040101,弁当類,498,374
P040101010,04,惣菜,0401,御飯類,040101,弁当類,580,435


---
> P-094: 先に作成したカテゴリ名付き商品データを以下の仕様でファイル出力せよ。なお、出力先のパスはdata配下とする。
>
> - ファイル形式はCSV（カンマ区切り）
> - ヘッダ有り
> - 文字コードはUTF-8

In [124]:
using System.IO;

using (StreamWriter sw = new StreamWriter(@"P_df_product_full_UTF-8_header.csv"))
using (CsvWriter csv = new CsvWriter(sw, CultureInfo.InvariantCulture))
{
    csv.WriteRecords(df_product_full);
}

---
> P-095: 先に作成したカテゴリ名付き商品データを以下の仕様でファイル出力せよ。なお、出力先のパスはdata配下とする。
>
> - ファイル形式はCSV（カンマ区切り）
> - ヘッダ有り
> - 文字コードはCP932

In [125]:
#r "nuget:System.Text.Encoding.CodePages"

using System.Text;
Encoding.RegisterProvider(CodePagesEncodingProvider.Instance);

Installed Packages System.Text.Encoding.CodePages, 9.0.9

In [126]:
using System.IO;
using System.Text;

using (StreamWriter sw = new StreamWriter(@"P_df_product_full_CP932_header.csv", false, Encoding.GetEncoding("Shift_JIS")))
using (CsvWriter csv = new CsvWriter(sw, CultureInfo.InvariantCulture))
{
    csv.Configuration.HasHeaderRecord = true;
    csv.WriteRecords(df_product_full);
}

---
> P-096: 先に作成したカテゴリ名付き商品データを以下の仕様でファイル出力せよ。なお、出力先のパスはdata配下とする。
>
> - ファイル形式はCSV（カンマ区切り）
> - ヘッダ無し
> - 文字コードはUTF-8

In [127]:
using System.IO;

using (StreamWriter sw = new StreamWriter(@"P_df_product_full_UTF-8_noh.csv"))
using (CsvWriter csv = new CsvWriter(sw, CultureInfo.InvariantCulture))
{
    csv.Configuration.HasHeaderRecord = false;
    csv.WriteRecords(df_product_full);
}

---
> P-097: 先に作成した以下形式のファイルを読み込み、データフレームを作成せよ。また、先頭10件を表示させ、正しくとりまれていることを確認せよ。
>
> - ファイル形式はCSV（カンマ区切り）
> - ヘッダ有り
> - 文字コードはUTF-8

In [129]:
var fileName = "P_df_product_full_UTF-8_header.csv";
List<ProductFull> ans097;
using (var reader = new StreamReader(fileName))
    using (var csv = new CsvReader(reader, CultureInfo.InvariantCulture))
    {
        csv.Configuration.HasHeaderRecord = true;
        var records = csv.GetRecords<ProductFull>();
        ans097 = records.ToList();
    }

ans097.Take(10).DisplayTable()

product_cd,category_major_cd,category_major_name,category_medium_cd,category_medium_name,category_small_cd,category_small_name,unit_price,unit_cost
P040101001,04,惣菜,0401,御飯類,040101,弁当類,198,149
P040101002,04,惣菜,0401,御飯類,040101,弁当類,218,164
P040101003,04,惣菜,0401,御飯類,040101,弁当類,230,173
P040101004,04,惣菜,0401,御飯類,040101,弁当類,248,186
P040101005,04,惣菜,0401,御飯類,040101,弁当類,268,201
P040101006,04,惣菜,0401,御飯類,040101,弁当類,298,224
P040101007,04,惣菜,0401,御飯類,040101,弁当類,338,254
P040101008,04,惣菜,0401,御飯類,040101,弁当類,420,315
P040101009,04,惣菜,0401,御飯類,040101,弁当類,498,374
P040101010,04,惣菜,0401,御飯類,040101,弁当類,580,435


---
> P-098: 先に作成した以下形式のファイルを読み込み、データフレームを作成せよ。また、先頭10件を表示させ、正しくとりまれていることを確認せよ。
>
> - ファイル形式はCSV（カンマ区切り）
> - ヘッダ無し
> - 文字コードはUTF-8

In [130]:
var fileName = "P_df_product_full_UTF-8_noh.csv";
List<ProductFull> ans098;
using (var reader = new StreamReader(fileName))
    using (var csv = new CsvReader(reader, CultureInfo.InvariantCulture))
    {
        csv.Configuration.HasHeaderRecord = false;
        var records = csv.GetRecords<ProductFull>();
        ans098 = records.ToList();
    }

ans098.Take(10).DisplayTable()

product_cd,category_major_cd,category_major_name,category_medium_cd,category_medium_name,category_small_cd,category_small_name,unit_price,unit_cost
P040101001,04,惣菜,0401,御飯類,040101,弁当類,198,149
P040101002,04,惣菜,0401,御飯類,040101,弁当類,218,164
P040101003,04,惣菜,0401,御飯類,040101,弁当類,230,173
P040101004,04,惣菜,0401,御飯類,040101,弁当類,248,186
P040101005,04,惣菜,0401,御飯類,040101,弁当類,268,201
P040101006,04,惣菜,0401,御飯類,040101,弁当類,298,224
P040101007,04,惣菜,0401,御飯類,040101,弁当類,338,254
P040101008,04,惣菜,0401,御飯類,040101,弁当類,420,315
P040101009,04,惣菜,0401,御飯類,040101,弁当類,498,374
P040101010,04,惣菜,0401,御飯類,040101,弁当類,580,435


---
> P-099: 先に作成したカテゴリ名付き商品データを以下の仕様でファイル出力せよ。なお、出力先のパスはdata配下とする。
>
> - ファイル形式はTSV（タブ区切り）
> - ヘッダ有り
> - 文字コードはUTF-8

In [131]:
//df_product_full.to_csv('../data/P_df_product_full_UTF-8_header.tsv', sep='\t', encoding='UTF-8', index=False)

using System.IO;
using CsvHelper.Configuration;

// CSV 出力定義（タブ区切り）
CsvConfiguration conf = new CsvConfiguration(CultureInfo.InvariantCulture)
{
    Delimiter = "\t"
};

using (StreamWriter sw = new StreamWriter(@"P_df_product_full_UTF-8_header.tsv"))
using (CsvWriter csv = new CsvWriter(sw, conf))
{
    csv.WriteRecords(df_product_full);
}

---
> P-100: 先に作成した以下形式のファイルを読み込み、データフレームを作成せよ。また、先頭10件を表示させ、正しくとりまれていることを確認せよ。
>
> - ファイル形式はTSV（タブ区切り）
> - ヘッダ有り
> - 文字コードはUTF-8

In [132]:
using System.IO;
using CsvHelper.Configuration;

//df_tmp = pd.read_table('../data/P_df_product_full_UTF-8_header.tsv', encoding='UTF-8')
var fileName = "P_df_product_full_UTF-8_header.tsv";

// CSV 読み込み定義
CsvConfiguration conf = new CsvConfiguration(CultureInfo.InvariantCulture)
{
    Delimiter = "\t"
};

List<ProductFull> ans100;
using (var reader = new StreamReader(fileName))
    using (var csv = new CsvReader(reader, conf))
    {
        var records = csv.GetRecords<ProductFull>();
        ans100 = records.ToList();
    }

ans100.Take(10).DisplayTable()

product_cd,category_major_cd,category_major_name,category_medium_cd,category_medium_name,category_small_cd,category_small_name,unit_price,unit_cost
P040101001,04,惣菜,0401,御飯類,040101,弁当類,198,149
P040101002,04,惣菜,0401,御飯類,040101,弁当類,218,164
P040101003,04,惣菜,0401,御飯類,040101,弁当類,230,173
P040101004,04,惣菜,0401,御飯類,040101,弁当類,248,186
P040101005,04,惣菜,0401,御飯類,040101,弁当類,268,201
P040101006,04,惣菜,0401,御飯類,040101,弁当類,298,224
P040101007,04,惣菜,0401,御飯類,040101,弁当類,338,254
P040101008,04,惣菜,0401,御飯類,040101,弁当類,420,315
P040101009,04,惣菜,0401,御飯類,040101,弁当類,498,374
P040101010,04,惣菜,0401,御飯類,040101,弁当類,580,435


# これで１００本終わりです。おつかれさまでした！